In [6]:
import sys

!{sys.executable} -m pip install pandas

  Using cached pandas-2.3.3-cp310-cp310-win_amd64.whl (11.3 MB)
     ---------------------------------------- 0.0/348.2 kB ? eta -:--:--
     -------------------------------------  348.2/348.2 kB 7.2 MB/s eta 0:00:01
     -------------------------------------- 348.2/348.2 kB 7.2 MB/s eta 0:00:00
     ---------------------------------------- 0.0/508.3 kB ? eta -:--:--
     ----------------------------------- - 481.3/508.3 kB 10.0 MB/s eta 0:00:01
     -------------------------------------- 508.3/508.3 kB 7.9 MB/s eta 0:00:00
  Using cached numpy-2.2.6-cp310-cp310-win_amd64.whl (12.9 MB)



[notice] A new release of pip is available: 23.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import pandas as pd

print("Pandas version:", pd.__version__)

Pandas version: 2.3.3


In [2]:
DATA_PATH = "../data/raw/twcs/twcs.csv"


print(DATA_PATH)

../data/raw/twcs/twcs.csv


In [3]:
df_sample = pd.read_csv(DATA_PATH, nrows=10)

df_sample

,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id
0,1,sprintcare,False,Tue Oct 31 22:10:47 +0000 2017,@115712 I understand. I would like to assist y...,2,3.0
1,2,115712,True,Tue Oct 31 22:11:45 +0000 2017,@sprintcare and how do you propose we do that,NaN,1.0
2,3,115712,True,Tue Oct 31 22:08:27 +0000 2017,@sprintcare I have sent several private messag...,1,4.0
3,4,sprintcare,False,Tue Oct 31 21:54:49 +0000 2017,@115712 Please send us a Private Message so th...,3,5.0
4,5,115712,True,Tue Oct 31 21:49:35 +0000 2017,@sprintcare I did.,4,6.0
5,6,sprintcare,False,Tue Oct 31 21:46:24 +0000 2017,@115712 Can you please send us a private messa...,"5,7",8.0
6,8,115712,True,Tue Oct 31 21:45:10 +0000 2017,@sprintcare is the worst customer service,"9,6,10",NaN
7,11,sprintcare,False,Tue Oct 31 22:10:35 +0000 2017,@115713 This is saddening to hear. Please shoo...,NaN,12.0
8,12,115713,True,Tue Oct 31 22:04:47 +0000 2017,@sprintcare You gonna magically change your co...,"11,13,14",15.0
9,15,sprintcare,False,Tue Oct 31 20:03:31 +0000 2017,@115713 We understand your concerns and we'd l...,12,16.0


In [4]:
print(df_sample.columns.tolist())

['tweet_id', 'author_id', 'inbound', 'created_at', 'text', 'response_tweet_id', 'in_response_to_tweet_id']


In [5]:
df_sample.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   tweet_id                 10 non-null     int64  
 1   author_id                10 non-null     object 
 2   inbound                  10 non-null     bool   
 3   created_at               10 non-null     object 
 4   text                     10 non-null     object 
 5   response_tweet_id        8 non-null      object 
 6   in_response_to_tweet_id  9 non-null      float64
dtypes: bool(1), float64(1), int64(1), object(4)
memory usage: 618.0+ bytes


In [6]:
from collections import Counter

author_counts = Counter()

for chunk in pd.read_csv(DATA_PATH, usecols=["author_id"], chunksize=100_000):
    author_counts.update(chunk["author_id"].dropna())

print("Unique authors:", len(author_counts))

Unique authors: 702777


In [7]:
top_authors = author_counts.most_common(50)

for author, count in top_authors:
    print(f"{author:30} {count:,}")

AmazonHelp                     169,840
AppleSupport                   106,860
Uber_Support                   56,270
SpotifyCares                   43,265
Delta                          42,253
Tesco                          38,573
AmericanAir                    36,764
TMobileHelp                    34,317
comcastcares                   33,031
British_Airways                29,361
SouthwestAir                   28,977
VirginTrains                   27,817
Ask_Spectrum                   25,860
XboxSupport                    24,557
sprintcare                     22,381
hulu_support                   21,872
sainsburys                     19,466
GWRHelp                        19,364
AskPlayStation                 19,098
ChipotleTweets                 18,749
VerizonSupport                 17,966
UPSHelp                        17,817
ATVIAssist                     17,650
O2                             16,212
Safaricom_Care                 16,077
idea_cares                     15,724
AskTarget 

In [9]:
brand_names = [
    "AmazonHelp",
    "AppleSupport",
    "Uber_Support",
    "SpotifyCares",
    "Delta",
    "Tesco",
    "AmericanAir",
    "TMobileHelp",
    "comcastcares",
    "British_Airways"
]

brand_stats = {
    brand: {"inbound": 0, "outbound": 0}
    for brand in brand_names
}

for chunk in pd.read_csv(
    DATA_PATH,
    usecols=["author_id", "inbound"],
    chunksize=100_000
):
    for brand in brand_names:
        rows = chunk[chunk["author_id"] == brand]

        brand_stats[brand]["inbound"] += int((rows["inbound"] == True).sum())
        brand_stats[brand]["outbound"] += int((rows["inbound"] == False).sum())

for brand, stats in brand_stats.items():
    print(
        f"{brand:20} "
        f"inbound={stats['inbound']:>8,} "
        f"outbound={stats['outbound']:>8,}"
    )

AmazonHelp           inbound=       0 outbound= 169,840
AppleSupport         inbound=       0 outbound= 106,860
Uber_Support         inbound=       0 outbound=  56,270
SpotifyCares         inbound=       0 outbound=  43,265
Delta                inbound=       0 outbound=  42,253
Tesco                inbound=       0 outbound=  38,573
AmericanAir          inbound=       0 outbound=  36,764
TMobileHelp          inbound=       0 outbound=  34,317
comcastcares         inbound=       0 outbound=  33,031
British_Airways      inbound=       0 outbound=  29,361


In [11]:
candidate_brands = [
    "AmazonHelp",
    "AppleSupport",
    "Uber_Support",
    "SpotifyCares",
    "Delta",
    "Tesco",
    "AmericanAir",
    "TMobileHelp",
    "comcastcares",
    "British_Airways"
]

print("Candidate brands:", len(candidate_brands))

Candidate brands: 10


In [12]:
# Step 2: Find customer messages connected to our candidate brands

brand_tweet_to_brand = {}

# Pass 1: collect tweet IDs written by our candidate brands
for chunk in pd.read_csv(
    DATA_PATH,
    usecols=["tweet_id", "author_id"],
    chunksize=100_000
):
    brand_rows = chunk[chunk["author_id"].isin(candidate_brands)]

    for tweet_id, author in zip(
        brand_rows["tweet_id"],
        brand_rows["author_id"]
    ):
        brand_tweet_to_brand[tweet_id] = author

print("Brand tweets collected:", len(brand_tweet_to_brand))

Brand tweets collected: 590534


In [15]:
from collections import Counter

customer_message_counts = Counter()

def get_brand(tweet_id):
    """Return the brand if tweet_id belongs to one of our brands."""
    if pd.isna(tweet_id):
        return None

    try:
        tweet_id = int(float(tweet_id))
    except (ValueError, TypeError):
        return None

    return brand_tweet_to_brand.get(tweet_id)


# Pass 2: find inbound/customer tweets connected to a brand tweet
for chunk in pd.read_csv(
    DATA_PATH,
    usecols=["inbound", "in_response_to_tweet_id", "response_tweet_id"],
    chunksize=100_000,
    dtype={
        "in_response_to_tweet_id": "string",
        "response_tweet_id": "string"
    }
):
    inbound_rows = chunk[chunk["inbound"] == True]

    for previous_id, next_id in zip(
        inbound_rows["in_response_to_tweet_id"],
        inbound_rows["response_tweet_id"]
    ):
        brand = get_brand(previous_id)

        if brand:
            customer_message_counts[brand] += 1

        brand = get_brand(next_id)

        if brand:
            customer_message_counts[brand] += 1


print("Customer messages connected to each brand:")

for brand, count in customer_message_counts.most_common():
    print(f"{brand:30} {count:,}")

Customer messages connected to each brand:
AmazonHelp                     236,724
AppleSupport                   135,234
Uber_Support                   73,750
SpotifyCares                   54,284
AmericanAir                    50,422
TMobileHelp                    43,441
Delta                          43,235
comcastcares                   35,451
British_Airways                28,864
Tesco                          28,488


In [16]:
amazon_messages = []

for chunk in pd.read_csv(
    DATA_PATH,
    usecols=[
        "tweet_id",
        "author_id",
        "inbound",
        "created_at",
        "text",
        "response_tweet_id",
        "in_response_to_tweet_id"
    ],
    chunksize=100_000
):
    inbound_rows = chunk[chunk["inbound"] == True]

    for _, row in inbound_rows.iterrows():

        previous_id = row["in_response_to_tweet_id"]
        next_id = row["response_tweet_id"]

        previous_brand = get_brand(previous_id)
        next_brand = get_brand(next_id)

        if previous_brand == "AmazonHelp" or next_brand == "AmazonHelp":
            amazon_messages.append(row)

amazon_df = pd.DataFrame(amazon_messages)

print("AmazonHelp customer messages:", len(amazon_df))

AmazonHelp customer messages: 176756


In [17]:
amazon_df[["created_at", "text"]].head(20)

,created_at,text
182,Wed Nov 22 09:24:30 +0000 2017,@AmazonHelp ありがとうございます。\n今、電話で主人が対応していただいてます。
183,Wed Nov 22 09:30:36 +0000 2017,@AmazonHelp 電話で対応してもらいましたが改良されませんでした。\n保証期間も過ぎ...
185,Wed Nov 22 09:44:04 +0000 2017,@AmazonHelp こちらこそありがとうございました。
187,Wed Nov 22 09:14:39 +0000 2017,amazonのfireTVstickが見れない😢
235,Wed Nov 22 08:55:35 +0000 2017,amazonプライムビデオ、再生エラーが多いです
322,Tue Oct 31 23:22:08 +0000 2017,@AmazonHelp 3 different people have given 3 di...
324,Tue Oct 31 23:32:26 +0000 2017,@AmazonHelp I frankly don't have the patience ...
325,Tue Oct 31 22:16:32 +0000 2017,Way to drop the ball on customer service @1158...
327,Tue Oct 31 22:19:34 +0000 2017,@115823 I want my amazon payments account CLOS...
329,Tue Oct 31 22:32:07 +0000 2017,"@AmazonHelp Okay, danke für die Info"


In [18]:
amazon_df.iloc[0]

tweet_id                                                             270
author_id                                                         115770
inbound                                                             True
created_at                                Wed Nov 22 09:24:30 +0000 2017
text                       @AmazonHelp ありがとうございます。\n今、電話で主人が対応していただいてます。
response_tweet_id                                                    NaN
in_response_to_tweet_id                                            269.0
Name: 182, dtype: object

In [19]:
print(amazon_df.iloc[0]["tweet_id"])
print(amazon_df.iloc[0]["in_response_to_tweet_id"])
print(amazon_df.iloc[0]["response_tweet_id"])

270
269.0
nan


In [20]:
tweet_id = 269

for chunk in pd.read_csv(
    DATA_PATH,
    usecols=[
        "tweet_id",
        "author_id",
        "inbound",
        "created_at",
        "text",
        "response_tweet_id",
        "in_response_to_tweet_id"
    ],
    chunksize=100_000
):
    result = chunk[chunk["tweet_id"] == tweet_id]

    if not result.empty:
        display(result)
        break

,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id
181,269,AmazonHelp,False,Wed Nov 22 09:23:01 +0000 2017,@115770 こんにちは、アマゾン公式です。Fire TV Stickが見れないというのは...,"270,271",272.0


In [21]:
tweet_id = 271

for chunk in pd.read_csv(
    DATA_PATH,
    usecols=[
        "tweet_id",
        "author_id",
        "inbound",
        "created_at",
        "text",
        "response_tweet_id",
        "in_response_to_tweet_id"
    ],
    chunksize=100_000
):
    result = chunk[chunk["tweet_id"] == tweet_id]

    if not result.empty:
        display(result)
        break

,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id
183,271,115770,True,Wed Nov 22 09:30:36 +0000 2017,@AmazonHelp 電話で対応してもらいましたが改良されませんでした。\n保証期間も過ぎ...,273,269.0


In [22]:
import pandas as pd
from pathlib import Path

OUTPUT_PATH = "../data/processed/amazonhelp_tweets.csv"

amazon_rows = []

for chunk in pd.read_csv(
    DATA_PATH,
    usecols=[
        "tweet_id",
        "author_id",
        "inbound",
        "created_at",
        "text",
        "response_tweet_id",
        "in_response_to_tweet_id"
    ],
    chunksize=100_000
):
    
    # Keep tweets written by AmazonHelp
    brand_rows = chunk[chunk["author_id"] == "AmazonHelp"]

    # Keep customer tweets directly connected to AmazonHelp tweets
    customer_rows = chunk[
        (chunk["inbound"] == True) &
        (
            chunk["in_response_to_tweet_id"].isin(brand_tweet_to_brand.keys()) |
            chunk["response_tweet_id"].isin(brand_tweet_to_brand.keys())
        )
    ]

    selected = pd.concat([brand_rows, customer_rows])

    if not selected.empty:
        amazon_rows.append(selected)

amazon_df = pd.concat(amazon_rows, ignore_index=True)

amazon_df = amazon_df.drop_duplicates(subset="tweet_id")

print("AmazonHelp tweets collected:", len(amazon_df))

amazon_df.to_csv(OUTPUT_PATH, index=False)

print("Saved to:", OUTPUT_PATH)

AmazonHelp tweets collected: 422637
Saved to: ../data/processed/amazonhelp_tweets.csv


In [23]:
amazon_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 422637 entries, 0 to 422636
Data columns (total 7 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   tweet_id                 422637 non-null  int64  
 1   author_id                422637 non-null  object 
 2   inbound                  422637 non-null  bool   
 3   created_at               422637 non-null  object 
 4   text                     422637 non-null  object 
 5   response_tweet_id        253291 non-null  object 
 6   in_response_to_tweet_id  422084 non-null  float64
dtypes: bool(1), float64(1), int64(1), object(4)
memory usage: 19.7+ MB


In [24]:
amazon_df["inbound"].value_counts()

inbound
True     252797
False    169840
Name: count, dtype: int64

In [25]:
amazon_df[["author_id", "inbound", "text"]].head(20)

,author_id,inbound,text
0,AmazonHelp,False,@115770 こんにちは、アマゾン公式です。Fire TV Stickが見れないというのは...
1,AmazonHelp,False,@115770 カスタマーサービスにてお問い合わせ済みとのことで、お手数をおかけいたしました...
2,AmazonHelp,False,@115770 恐れ入ります。至らない点も多々あるかとは存じますが、今後ともどうぞよろしくお...
3,AmazonHelp,False,@115792 ご不便をおかけしております。アプリをご利用でしょうか。強制停止&gt;端末の...
4,AmazonHelp,False,@115820 I'm sorry we've let you down! Without ...
5,AmazonHelp,False,@115820 We'd like to take a further look into ...
6,AmazonHelp,False,@115822 I am unable to affect your account via...
7,AmazonHelp,False,"@115824 Hi, wir erhalten die Filme/Serien so v..."
8,AmazonHelp,False,@115824 Wir haben zu danken. Schönen Abend noc...
9,AmazonHelp,False,@115826 I'm sorry for the wait. You'll receive...


In [26]:
# Build a quick lookup of tweet_id -> tweet information

tweet_lookup = amazon_df.set_index("tweet_id").to_dict("index")

pairs = []

for _, customer in amazon_df[amazon_df["inbound"] == True].head(1000).iterrows():
    
    parent_id = customer["in_response_to_tweet_id"]
    
    if pd.isna(parent_id):
        continue
    
    parent_id = int(parent_id)
    
    parent = tweet_lookup.get(parent_id)
    
    if parent and parent["author_id"] == "AmazonHelp":
        pairs.append({
            "customer_text": customer["text"],
            "agent_text": parent["text"],
            "customer_tweet_id": customer["tweet_id"],
            "agent_tweet_id": parent_id
        })

pairs_df = pd.DataFrame(pairs)

print("Customer → AmazonHelp pairs found:", len(pairs_df))

Customer → AmazonHelp pairs found: 290


In [27]:
pairs_df[["customer_text", "agent_text"]].head(10)

,customer_text,agent_text
0,@AmazonHelp ありがとうございます。\n今、電話で主人が対応していただいてます。,@115770 こんにちは、アマゾン公式です。Fire TV Stickが見れないというのは...
1,@AmazonHelp 電話で対応してもらいましたが改良されませんでした。\n保証期間も過ぎ...,@115770 こんにちは、アマゾン公式です。Fire TV Stickが見れないというのは...
2,@AmazonHelp こちらこそありがとうございました。,@115770 カスタマーサービスにてお問い合わせ済みとのことで、お手数をおかけいたしました...
3,@AmazonHelp 3 different people have given 3 di...,@115820 I'm sorry we've let you down! Without ...
4,@AmazonHelp I frankly don't have the patience ...,@115820 We'd like to take a further look into ...
5,"@AmazonHelp Okay, danke für die Info","@115824 Hi, wir erhalten die Filme/Serien so v..."
6,@AmazonHelp @115826 Yeah this is crazy we’re l...,@115826 I'm sorry for the wait. You'll receive...
7,@AmazonHelp Hi ready for some help,"@115834 The Echo Show is supported, please rea..."
8,@AmazonHelp Nothing there helped me with the E...,"@115834 Oh no, I'm sorry for the issues! For t..."
9,@AmazonHelp That page is useless - doesn’t all...,@115835 I'm so sorry you didn't receive your p...


In [28]:
print("Total AmazonHelp tweets:", len(amazon_df))

print(
    "Customer tweets:",
    (amazon_df["inbound"] == True).sum()
)

print(
    "AmazonHelp tweets:",
    (amazon_df["inbound"] == False).sum()
)

print(
    "Direct customer → AmazonHelp pairs in sample:",
    len(pairs_df)
)

Total AmazonHelp tweets: 422637
Customer tweets: 252797
AmazonHelp tweets: 169840
Direct customer → AmazonHelp pairs in sample: 290


In [29]:
pairs_df.head(20).to_string(index=False)



'                                                                                                                                              customer_text                                                                                                                             agent_text  customer_tweet_id  agent_tweet_id\n                                                                                                              @AmazonHelp ありがとうございます。\\n今、電話で主人が対応していただいてます。         @115770 こんにちは、アマゾン公式です。Fire TV Stickが見れないというのは、どのような状況でしょうか。一般的なトラブルシューティングを記載したヘルプがございますので、ご参照ください。https://t.co/2pbG55qJ7h ET                270             269\n                                                                                           @AmazonHelp 電話で対応してもらいましたが改良されませんでした。\\n保証期間も過ぎてるので買い直しになるんでしょうね。         @115770 こんにちは、アマゾン公式です。Fire TV Stickが見れないというのは、どのような状況でしょうか。一般的なトラブルシューティングを記載したヘルプがございますので、ご参照ください。https://t.co/2pbG55qJ7h ET                271             269\n                

In [30]:
# Take a random sample of AmazonHelp customer messages
# so we can manually discover common intent categories.

intent_sample = amazon_df[
    amazon_df["inbound"] == True
].sample(
    n=200,
    random_state=42
)

intent_sample[["text"]].to_csv(
    "../data/processed/amazonhelp_intent_sample.csv",
    index=False
)

print("Saved 200 customer messages for intent analysis.")

Saved 200 customer messages for intent analysis.


In [31]:
pd.set_option("display.max_colwidth", 300)

intent_sample[["text"]].head(50)

,text
17046,"@British_Airways Hail, ‘Oh Happy to Hbelp’ Charlotte. Any chance of a reply? Finally on way home with TAP who are polite, helpful and efficient, unlike BA. I will be claiming maximum compensation from your dreadful company for this debacle. Ian Manning __email__ 07904 576301 BH53EH"
385512,@British_Airways I would call your Customer Relations department to get that information I sent numerous times but they are only open 6am-10am PST Mon-Fri. Did you know that 55 million people live in that time zone? #BritishAirways #CustomerService #MondayMotivation #WhoCares
135608,@AmazonHelp Help would be when you get in contact with the delivery team and see what is happening
346835,"@British_Airways You put the box on the main page and not at the bottom of the T&amp;Cs which people who are in a rush and don’t have time to read an epistle just tick without looking. It’s wrong, you know it’s wrong. We paid for a flight that you’ve sold from under us without telling us. Disgus..."
121841,"@AmazonHelp Era hoy, ahora es mañana, pero como he dicho no voy a perder un día entero esperando, si hay suerte bien, si no se cancela el pedido."
227343,@SpotifyCares If I reinstall will I have to download all my songs again?
383304,"@British_Airways Thank you, I appreciate that. Please could you tell me how I can put in a complaint and for compensation?"
192436,@SpotifyCares I'm using a Lumia 950 running Windows 10 Mobile (latest version) and the latest Spotify app for Windows Phone / 10 Mobile.
158681,"@Tesco Hi Danny, I could future just not go to Tesco."
111881,@AmazonHelp A partir de cuándo estará a la venta?


In [32]:
intent_sample = intent_sample.copy()

intent_sample["intent"] = ""

intent_sample[
    ["text", "intent"]
].to_csv(
    "../data/processed/amazonhelp_intent_labeling.csv",
    index=False
)

print("Labeling file created.")
print("Rows to label:", len(intent_sample))

Labeling file created.
Rows to label: 200


In [33]:
label_batch = intent_sample[["text", "intent"]].reset_index(drop=True)

label_batch.iloc[0:25]

,text,intent
0,"@British_Airways Hail, ‘Oh Happy to Hbelp’ Charlotte. Any chance of a reply? Finally on way home with TAP who are polite, helpful and efficient, unlike BA. I will be claiming maximum compensation from your dreadful company for this debacle. Ian Manning __email__ 07904 576301 BH53EH",
1,@British_Airways I would call your Customer Relations department to get that information I sent numerous times but they are only open 6am-10am PST Mon-Fri. Did you know that 55 million people live in that time zone? #BritishAirways #CustomerService #MondayMotivation #WhoCares,
2,@AmazonHelp Help would be when you get in contact with the delivery team and see what is happening,
3,"@British_Airways You put the box on the main page and not at the bottom of the T&amp;Cs which people who are in a rush and don’t have time to read an epistle just tick without looking. It’s wrong, you know it’s wrong. We paid for a flight that you’ve sold from under us without telling us. Disgus...",
4,"@AmazonHelp Era hoy, ahora es mañana, pero como he dicho no voy a perder un día entero esperando, si hay suerte bien, si no se cancela el pedido.",
5,@SpotifyCares If I reinstall will I have to download all my songs again?,
6,"@British_Airways Thank you, I appreciate that. Please could you tell me how I can put in a complaint and for compensation?",
7,@SpotifyCares I'm using a Lumia 950 running Windows 10 Mobile (latest version) and the latest Spotify app for Windows Phone / 10 Mobile.,
8,"@Tesco Hi Danny, I could future just not go to Tesco.",
9,@AmazonHelp A partir de cuándo estará a la venta?,


In [34]:
amazon_tweet_ids = set(
    amazon_df.loc[
        amazon_df["author_id"] == "AmazonHelp",
        "tweet_id"
    ]
)

print("AmazonHelp tweet IDs:", len(amazon_tweet_ids))

AmazonHelp tweet IDs: 169840


In [35]:
# Re-extract only customer tweets directly connected to AmazonHelp

amazon_customer_rows = []

for chunk in pd.read_csv(
    DATA_PATH,
    usecols=[
        "tweet_id",
        "author_id",
        "inbound",
        "created_at",
        "text",
        "response_tweet_id",
        "in_response_to_tweet_id"
    ],
    chunksize=100_000
):
    customer_rows = chunk[chunk["inbound"] == True].copy()

    # Only use the tweet the customer is directly replying to.
    customer_rows = customer_rows[
        customer_rows["in_response_to_tweet_id"].isin(
            amazon_tweet_ids
        )
    ]

    if not customer_rows.empty:
        amazon_customer_rows.append(customer_rows)

amazon_customers = pd.concat(
    amazon_customer_rows,
    ignore_index=True
).drop_duplicates("tweet_id")

print("AmazonHelp customer messages:", len(amazon_customers))

AmazonHelp customer messages: 100503


In [36]:
print(
    amazon_customers["author_id"].head(20).tolist()
)

print(
    "Unique customer authors:",
    amazon_customers["author_id"].nunique()
)

['115770', '115770', '115770', '115820', '115820', '115824', '115827', '115834', '115834', '115835', '115838', '115838', '115839', '115843', '115844', '115849', '115849', '116081', '116081', '116082']
Unique customer authors: 40671


In [37]:
intent_sample = amazon_customers.sample(
    n=200,
    random_state=42
).reset_index(drop=True)

intent_sample["intent"] = ""

print("Intent discovery sample:", len(intent_sample))

Intent discovery sample: 200


In [38]:
pd.set_option("display.max_colwidth", 300)

intent_sample.iloc[0:25][["text", "intent"]]

,text,intent
0,@AmazonHelp Something was supposed to be delivered today but it says on the order page ‘arrival at incorrect carrier facility’ and to ‘check back tuesday’ (yesterday),
1,@AmazonHelp Thank you 🙌🏻,
2,@AmazonHelp C’est Amazon Logistics,
3,"@AmazonHelp yes I have been through all of the steps. I will check with nighbours further down the street tomorrow, but no notification of delivery 1/2",
4,@AmazonHelp But kindly help me to get it faster and help me to get it on 16 . I a k so know promise date but can't you do something,
5,"@AmazonHelp Well you couldn't resolve the 'issue'"" your colleagues messed me around and even your superiors did. I genuinely feel worthless to you. :(",
6,@AmazonHelp Jk I figured it out ty tho,
7,@AmazonHelp Voilà j'ai envoyé en message privé,
8,@AmazonHelp Sick of ur response.No need of it.,
9,@AmazonHelp MAY I KNOW HOW MANY TIMES I NEED TO UPDATE..I AM NOY GOING TO UPDATE ANY MORE..IF U WANT TO CONTINUE ..U CAN ELSE CLOSE THIS DISCUSSION.,


In [39]:
labels_0_24 = [
    "delivery_issue",
    "follow_up_or_confirmation",
    "other",
    "delivery_issue",
    "delivery_date_change",
    "escalation_or_complaint",
    "follow_up_or_confirmation",
    "follow_up_or_confirmation",
    "escalation_or_complaint",
    "escalation_or_complaint",
    "delivery_issue",
    "follow_up_or_confirmation",
    "escalation_or_complaint",
    "follow_up_or_confirmation",
    "follow_up_or_confirmation",
    "delivery_issue",
    "follow_up_or_confirmation",
    "order_status",
    "delivery_date_change",
    "delivery_issue",
    "escalation_or_complaint",
    "product_availability",
    "order_status",
    "delivery_issue",
    "escalation_or_complaint"
]

intent_sample.loc[0:24, "intent"] = labels_0_24

print(intent_sample.loc[0:24, ["text", "intent"]].to_string(index=False))

                                                                                                                                                                                                                                                                                                text                    intent
                                                                                                                              @AmazonHelp Something was supposed to be delivered today but it says on the order page ‘arrival at incorrect carrier facility’ and to ‘check back tuesday’ (yesterday)            delivery_issue
                                                                                                                                                                                                                                                                            @AmazonHelp Thank you 🙌🏻 follow_up_or_confirmation
                                           

In [40]:
intent_sample["intent"].value_counts()

intent
                             175
follow_up_or_confirmation      7
escalation_or_complaint        6
delivery_issue                 6
order_status                   2
delivery_date_change           2
other                          1
product_availability           1
Name: count, dtype: int64

In [41]:
intent_sample.iloc[25:50][["text", "intent"]]

,text,intent
25,@AmazonHelp I’ve had multiple bad experiences this year and I’ve rolled with it but it’s getting utterly ridiculous. Phone support was no help at all,
26,@AmazonHelp Thanks! Just ordered a replacement!,
27,@AmazonHelp I am No longer a prime member,
28,@AmazonHelp My order didn’t ship until TWO DAYS after I ordered! #fail,
29,"@AmazonHelp 2nd day in a row. Do you not use safe places anymore. Why are people paying for a prime service, when we don't get one. https://t.co/IgOLLLbE6i",
30,@AmazonHelp I have messaged Amazon a couple of times and nobody bothered to pick it up.,
31,@AmazonHelp Beide Male 06.11. Nur eines ist angekommen. Bestellt letzten Mi abend und Do früh. Beim Kauf hieß es aber Prime Premiumversand zum 03.11.,
32,@AmazonHelp C’est bien parfois de montrer aux entreprises qui gerent qu’on le voit 👍🏽 pensez à le dire à ceux qui ne le voient pas (employés etc) 👍🏽☺️,
33,@AmazonHelp Nothing! It was 50 miles from me 3:50 am and no updates until after 8pm today,
34,@AmazonHelp Bt then u r refunding only 100rs if customer is self returning. Plz return full payment according to bill...unfair to customers,


In [42]:
labels_25_49 = [
    "escalation_or_complaint",
    "follow_up_or_confirmation",
    "prime_membership",
    "delivery_issue",
    "delivery_issue",
    "escalation_or_complaint",
    "delivery_issue",
    "follow_up_or_confirmation",
    "order_status",
    "refund",
    "product_availability",
    "delivery_issue",
    "other",
    "follow_up_or_confirmation",
    "delivery_issue",
    "delivery_issue",
    "order_status",
    "escalation_or_complaint",
    "order_status",
    "order_status",
    "delivery_issue",
    "refund",
    "order_status",
    "follow_up_or_confirmation",
    "follow_up_or_confirmation"
]

intent_sample.loc[25:49, "intent"] = labels_25_49

print(
    intent_sample.loc[25:49, ["text", "intent"]].to_string(index=False)
)

                                                                                                                                                                                               text                    intent
                                              @AmazonHelp I’ve had multiple bad experiences this year and I’ve rolled with it but it’s getting utterly ridiculous. Phone support was no help at all   escalation_or_complaint
                                                                                                                                                    @AmazonHelp Thanks! Just ordered a replacement! follow_up_or_confirmation
                                                                                                                                                          @AmazonHelp I am No longer a prime member          prime_membership
                                                                                                                

In [43]:
print(
    intent_sample["intent"]
    .replace("", pd.NA)
    .dropna()
    .value_counts()
)

intent
delivery_issue               13
follow_up_or_confirmation    12
escalation_or_complaint       9
order_status                  7
other                         2
delivery_date_change          2
product_availability          2
refund                        2
prime_membership              1
Name: count, dtype: int64


In [44]:
print(intent_sample.index.tolist()[50:75])

[50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74]


In [45]:
print(
    intent_sample.iloc[50:75]["text"].to_string()
)

50                                                                                                                                                                                       @AmazonHelp Still My Money not return back from AMAZON even after 4 days. Vahiyat Service by @115850 @115821 @AmazonHelp
51                                                                                                                                                                                                          @AmazonHelp No help at all because they are sold out in your store. So basically zero help whatsoever
52                                        @AmazonHelp Difficulté à avoir du stock.\nOr, @157182 annonce un réassort le 5 Décembre et jusqu'à présent on m'avait promis une livraison à cette période. Maintenant on annule à nouveau cela signifie que @157182  à donc menti (une fois de plus) sur le réassort ?
53                                                                                

In [46]:
labels_50_74 = [
    "refund",
    "product_availability",
    "product_availability",
    "delivery_issue",
    "delivery_issue",
    "delivery_issue",
    "escalation_or_complaint",
    "delivery_issue",
    "follow_up_or_confirmation",
    "order_status",
    "delivery_issue",
    "order_status",
    "delivery_issue",
    "order_status",
    "follow_up_or_confirmation",
    "follow_up_or_confirmation",
    "other",
    "other",
    "order_status",
    "follow_up_or_confirmation",
    "other",
    "delivery_issue",
    "return_or_cancellation",
    "delivery_issue",
    "escalation_or_complaint"
]

intent_sample.loc[50:74, "intent"] = labels_50_74

print(
    intent_sample["intent"]
    .replace("", pd.NA)
    .dropna()
    .value_counts()
)

intent
delivery_issue               21
follow_up_or_confirmation    16
order_status                 11
escalation_or_complaint      11
other                         5
product_availability          4
refund                        3
delivery_date_change          2
prime_membership              1
return_or_cancellation        1
Name: count, dtype: int64


In [47]:
print(
    intent_sample.iloc[75:100]["text"].to_string()
)

75                                                                                                                                             @AmazonHelp 4. the seller by trying to locate the Contact the Seller button on my order, only to find there wasn't one. I therefore tried to leave a &gt;
76                                                                                                                                                                                                                                                                         @AmazonHelp ありがとうございます＼(^o^)／
77                                                                                                                                                                                                @AmazonHelp This was a lightening deal...I will have to pay more for a reorder https://t.co/RvqEcIyVDd
78                                                                                                           

In [48]:
labels_75_99 = [
    "other",
    "follow_up_or_confirmation",
    "price_or_billing",
    "delivery_issue",
    "product_issue",
    "technical_support",
    "delivery_issue",
    "payment_or_account",
    "delivery_issue",
    "escalation_or_complaint",
    "escalation_or_complaint",
    "follow_up_or_confirmation",
    "delivery_issue",
    "follow_up_or_confirmation",
    "follow_up_or_confirmation",
    "escalation_or_complaint",
    "delivery_issue",
    "delivery_issue",
    "delivery_issue",
    "follow_up_or_confirmation",
    "delivery_issue",
    "order_status",
    "refund",
    "payment_or_account",
    "delivery_issue"
]

intent_sample.loc[75:99, "intent"] = labels_75_99

print(
    intent_sample["intent"]
    .replace("", pd.NA)
    .dropna()
    .value_counts()
)

intent
delivery_issue               30
follow_up_or_confirmation    21
escalation_or_complaint      14
order_status                 12
other                         6
product_availability          4
refund                        4
delivery_date_change          2
payment_or_account            2
prime_membership              1
return_or_cancellation        1
price_or_billing              1
product_issue                 1
technical_support             1
Name: count, dtype: int64


In [49]:
print(
    intent_sample.iloc[100:125]["text"].to_string()
)

100                                                                                                                                                                              @AmazonHelp Vendido por Amazon. Como lo solucionamos?
101                                                                                                                  @AmazonHelp Your apps consistently fail. I have to reset often. It’s right next to the router.  Just tired of it.
102                                                                                                                                                                                             @AmazonHelp Ups. Over an hour late now
103                                                                                                                                                       @AmazonHelp Will Amazon solve my problem or I will have to take legal action
104                                                                         

In [50]:
labels_100_124 = [
    "order_status",
    "technical_support",
    "delivery_issue",
    "escalation_or_complaint",
    "follow_up_or_confirmation",
    "order_status",
    "refund",
    "product_issue",
    "escalation_or_complaint",
    "escalation_or_complaint",
    "delivery_issue",
    "prime_membership",
    "technical_support",
    "order_status",
    "refund",
    "prime_membership",
    "escalation_or_complaint",
    "escalation_or_complaint",
    "escalation_or_complaint",
    "payment_or_account",
    "delivery_issue",
    "follow_up_or_confirmation",
    "other",
    "order_status",
    "prime_membership"
]

intent_sample.loc[100:124, "intent"] = labels_100_124

print(
    intent_sample["intent"]
    .replace("", pd.NA)
    .dropna()
    .value_counts()
)

intent
delivery_issue               33
follow_up_or_confirmation    23
escalation_or_complaint      20
order_status                 16
other                         7
refund                        6
prime_membership              4
product_availability          4
technical_support             3
payment_or_account            3
delivery_date_change          2
product_issue                 2
return_or_cancellation        1
price_or_billing              1
Name: count, dtype: int64


In [51]:
print(
    intent_sample.iloc[125:150]["text"].to_string()
)

125                                                                                                                                                                                @AmazonHelp  https://t.co/lrKiVl1Xas
126                                                                                                                                                                                    @AmazonHelp Wat is the status..?
127                                                                                                                                                                 @AmazonHelp L'article est bien vendu par Amazon ...
128                                                                                                                                                                          @AmazonHelp 31 a 1 https://t.co/0L22OSUXX0
129                                                              @AmazonHelp Thank u. I have read how 2 con-customers cheated Amazon of 

In [52]:
labels_125_149 = [
    "other",
    "order_status",
    "order_status",
    "other",
    "follow_up_or_confirmation",
    "escalation_or_complaint",
    "product_availability",
    "delivery_issue",
    "follow_up_or_confirmation",
    "product_issue",
    "order_status",
    "escalation_or_complaint",
    "prime_membership",
    "escalation_or_complaint",
    "product_availability",
    "follow_up_or_confirmation",
    "escalation_or_complaint",
    "follow_up_or_confirmation",
    "follow_up_or_confirmation",
    "technical_support",
    "delivery_issue",
    "technical_support",
    "product_availability",
    "return_or_cancellation",
    "return_or_cancellation"
]

intent_sample.loc[125:149, "intent"] = labels_125_149

print(
    intent_sample["intent"]
    .replace("", pd.NA)
    .dropna()
    .value_counts()
)

intent
delivery_issue               35
follow_up_or_confirmation    28
escalation_or_complaint      24
order_status                 19
other                         9
product_availability          7
refund                        6
prime_membership              5
technical_support             5
return_or_cancellation        3
payment_or_account            3
product_issue                 3
delivery_date_change          2
price_or_billing              1
Name: count, dtype: int64


In [54]:
print(
    intent_sample.iloc[150:175]["text"].to_string()
)

150                                                                                                                              @AmazonHelp I want refund so I can buy something else guys.. at least in Amazon wallet? This is really rediculous guys. I will not buy electronic product from @115850
151                                                                                                                                              @AmazonHelp Product was late to ship by amazon.ca. Contact Amazon Customer Service to request info and see if shipping date could be met, or I cancel.
152                               @AmazonHelp I don't care if its visible to public.\n\nIf things will not going on track, I'll share both the Chats here, for all the world and people can see how your executives do what they want and avoiding customers troubles.\n\n@115851 @115821 @115850 @5339
153                                                                                                             

In [56]:
labels_150_174 = [
    "refund",
    "delivery_issue",
    "escalation_or_complaint",
    "delivery_issue",
    "escalation_or_complaint",
    "follow_up_or_confirmation",
    "escalation_or_complaint",
    "delivery_issue",
    "delivery_issue",
    "escalation_or_complaint",
    "delivery_issue",
    "other",
    "price_or_billing",
    "delivery_issue",
    "prime_membership",
    "delivery_issue",
    "escalation_or_complaint",
    "order_status",
    "escalation_or_complaint",
    "payment_or_account",
    "payment_or_account",
    "follow_up_or_confirmation",
    "follow_up_or_confirmation",
    "return_or_cancellation",
    "escalation_or_complaint"
]

intent_sample.loc[150:174, "intent"] = labels_150_174

print(
    intent_sample["intent"]
    .replace("", pd.NA)
    .dropna()
    .value_counts()
)

intent
delivery_issue               42
follow_up_or_confirmation    31
escalation_or_complaint      31
order_status                 20
other                        10
product_availability          7
refund                        7
prime_membership              6
technical_support             5
payment_or_account            5
return_or_cancellation        4
product_issue                 3
delivery_date_change          2
price_or_billing              2
Name: count, dtype: int64


In [57]:
intent_sample.to_csv(
    "../data/processed/amazonhelp_intent_discovery_labeled.csv",
    index=False
)

print("Saved labeled intent dataset.")

Saved labeled intent dataset.


In [58]:
print(
    intent_sample[["tweet_id", "text", "intent"]]
    .head()
    .to_string(index=False)
)

 tweet_id                                                                                                                                                                   text                    intent
  2470126 @AmazonHelp Something was supposed to be delivered today but it says on the order page ‘arrival at incorrect carrier facility’ and to ‘check back tuesday’ (yesterday)            delivery_issue
  1705395                                                                                                                                               @AmazonHelp Thank you 🙌🏻 follow_up_or_confirmation
  2957011                                                                                                                                     @AmazonHelp C’est Amazon Logistics                     other
    55058                @AmazonHelp yes I have been through all of the steps. I will check with nighbours further down the street tomorrow, but no notification of delivery 1/2            

In [59]:
final_intent_mapping = {
    "delivery_issue": "delivery_issue",
    "delivery_date_change": "delivery_issue",
    "order_status": "order_status",
    "refund": "refund",
    "return_or_cancellation": "return_or_cancellation",
    "product_issue": "product_issue",
    "technical_support": "technical_support",
    "prime_membership": "prime_membership",
    "payment_or_account": "payment_or_billing",
    "price_or_billing": "payment_or_billing",
    "product_availability": "product_availability",
    "escalation_or_complaint": "complaint_or_escalation",
    "follow_up_or_confirmation": "other",
    "other": "other"
}

intent_sample["final_intent"] = intent_sample["intent"].map(
    final_intent_mapping
)

print(
    intent_sample["final_intent"].value_counts()
)

final_intent
delivery_issue             44
other                      41
complaint_or_escalation    31
order_status               20
product_availability        7
refund                      7
payment_or_billing          7
prime_membership            6
technical_support           5
return_or_cancellation      4
product_issue               3
Name: count, dtype: int64


In [60]:
final_intent_df = intent_sample[
    [
        "tweet_id",
        "author_id",
        "created_at",
        "text",
        "final_intent"
    ]
].copy()

final_intent_df.to_csv(
    "../data/processed/amazonhelp_intent_labeled.csv",
    index=False
)

print(
    "Saved:",
    len(final_intent_df),
    "labeled examples"
)

print(
    "Columns:",
    final_intent_df.columns.tolist()
)

Saved: 200 labeled examples
Columns: ['tweet_id', 'author_id', 'created_at', 'text', 'final_intent']


In [61]:
print(
    final_intent_df["final_intent"].value_counts()
)

final_intent
delivery_issue             44
other                      41
complaint_or_escalation    31
order_status               20
product_availability        7
refund                      7
payment_or_billing          7
prime_membership            6
technical_support           5
return_or_cancellation      4
product_issue               3
Name: count, dtype: int64


In [63]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [64]:
from baselines.majority import majority_class_predict

y = final_intent_df["final_intent"]

predictions = majority_class_predict(y, len(y))

print("Predicted class:", predictions[0])
print("Number of predictions:", len(predictions))

Predicted class: delivery_issue
Number of predictions: 200


In [66]:
import sys
!{sys.executable} -m pip install scikit-learn

  Using cached scikit_learn-1.7.2-cp310-cp310-win_amd64.whl (8.9 MB)
     ---------------------------------------- 0.0/306.1 kB ? eta -:--:--
     -------------------------------------- 306.1/306.1 kB 6.3 MB/s eta 0:00:00
  Using cached scipy-1.15.3-cp310-cp310-win_amd64.whl (41.3 MB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)



[notice] A new release of pip is available: 23.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [67]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y, predictions)

print(f"Majority baseline accuracy: {accuracy:.4f}")

TypeError: '<' not supported between instances of 'float' and 'str'

In [68]:
print(y.dtype)
print(y.unique())

object
['delivery_issue' 'other' 'complaint_or_escalation' 'order_status'
 'product_availability' 'prime_membership' 'refund'
 'return_or_cancellation' 'payment_or_billing' 'product_issue'
 'technical_support' nan]


In [69]:
print(y.isna().sum())

25


In [70]:
unlabeled = intent_sample[
    intent_sample["final_intent"].isna()
][["tweet_id", "text", "intent", "final_intent"]]

print(unlabeled.to_string(index=False))

 tweet_id                                                                                                                                                                                                                                                                                       text intent final_intent
   974395                                                                                                                                                                                                                    @AmazonHelp そうでしたか。アプリからはリスト上あがってくるのに視聴出来ないのは残念ですね。また再開されることを楽しみに待ってます。                 NaN
   729259                                                                                                                                                                                                                                                               @AmazonHelp grazie lo stesso                 NaN
  1631035                                                    

In [71]:
missing_labels = [
    "other",
    "other",
    "complaint_or_escalation",
    "product_issue",
    "other",
    "technical_support",
    "complaint_or_escalation",
    "other",
    "return_or_cancellation",
    "delivery_issue",
    "delivery_issue",
    "complaint_or_escalation",
    "delivery_issue",
    "delivery_issue",
    "delivery_issue",
    "payment_or_billing",
    "product_issue",
    "other",
    "other",
    "other",
    "other",
    "technical_support",
    "refund",
    "delivery_issue",
    "delivery_issue",
]

intent_sample.loc[
    intent_sample["final_intent"].isna(),
    "final_intent"
] = missing_labels

In [72]:
print("Missing labels:", intent_sample["final_intent"].isna().sum())
print(intent_sample["final_intent"].value_counts())

Missing labels: 0
final_intent
delivery_issue             51
other                      49
complaint_or_escalation    34
order_status               20
refund                      8
payment_or_billing          8
product_availability        7
technical_support           7
prime_membership            6
return_or_cancellation      5
product_issue               5
Name: count, dtype: int64


In [73]:
final_intent_df = intent_sample[
    [
        "tweet_id",
        "author_id",
        "created_at",
        "text",
        "final_intent"
    ]
].copy()

final_intent_df.to_csv(
    "../data/processed/amazonhelp_intent_labeled.csv",
    index=False
)

print("Saved:", len(final_intent_df), "labeled examples")

Saved: 200 labeled examples


In [74]:
from baselines.majority import majority_class_predict

y = final_intent_df["final_intent"]

predictions = majority_class_predict(y, len(y))

print("Predicted class:", predictions[0])
print("Number of predictions:", len(predictions))

Predicted class: delivery_issue
Number of predictions: 200


In [75]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y, predictions)

print(f"Majority baseline accuracy: {accuracy:.4f}")

Majority baseline accuracy: 0.2550


In [76]:
from baselines.tfidf_logistic import (
    train_tfidf_logistic,
    predict_tfidf_logistic
)

print("TF-IDF + Logistic Regression imported successfully")

TF-IDF + Logistic Regression imported successfully


In [77]:
from sklearn.model_selection import train_test_split

X = final_intent_df["text"]
y = final_intent_df["final_intent"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training examples:", len(X_train))
print("Test examples:", len(X_test))

Training examples: 160
Test examples: 40


In [78]:
vectorizer, model = train_tfidf_logistic(
    X_train,
    y_train
)

print("Model trained successfully")

Model trained successfully


In [79]:
predictions_lr = predict_tfidf_logistic(
    vectorizer,
    model,
    X_test
)

print("Predictions made:", len(predictions_lr))

Predictions made: 40


In [80]:
from sklearn.metrics import accuracy_score

lr_accuracy = accuracy_score(y_test, predictions_lr)

print(f"TF-IDF + Logistic Regression accuracy: {lr_accuracy:.4f}")

TF-IDF + Logistic Regression accuracy: 0.4500


In [81]:
from sklearn.metrics import accuracy_score

lr_accuracy = accuracy_score(y_test, predictions_lr)

print(f"TF-IDF + Logistic Regression accuracy: {lr_accuracy:.4f}")

TF-IDF + Logistic Regression accuracy: 0.4500


In [82]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test,
        predictions_lr,
        zero_division=0
    )
)

                         precision    recall  f1-score   support

complaint_or_escalation       0.00      0.00      0.00         7
         delivery_issue       0.41      0.90      0.56        10
           order_status       0.00      0.00      0.00         4
                  other       0.50      0.90      0.64        10
     payment_or_billing       0.00      0.00      0.00         2
       prime_membership       0.00      0.00      0.00         1
   product_availability       0.00      0.00      0.00         1
          product_issue       0.00      0.00      0.00         1
                 refund       0.00      0.00      0.00         2
 return_or_cancellation       0.00      0.00      0.00         1
      technical_support       0.00      0.00      0.00         1

               accuracy                           0.45        40
              macro avg       0.08      0.16      0.11        40
           weighted avg       0.23      0.45      0.30        40



In [83]:
import shutil

shutil.copy(
    "../data/processed/amazonhelp_intent_labeled.csv",
    "../data/golden/amazonhelp_golden.csv"
)

print("Golden evaluation set created.")

Golden evaluation set created.


In [84]:
golden_df = pd.read_csv(
    "../data/golden/amazonhelp_golden.csv"
)

print("Golden examples:", len(golden_df))
print("Missing labels:", golden_df["final_intent"].isna().sum())

Golden examples: 200
Missing labels: 0


In [2]:
# Get all AmazonHelp customer messages
all_customer_ids = set(amazon_customers["tweet_id"])

# Get the 200 golden-set IDs
golden_ids = set(golden_df["tweet_id"])

# Remove golden examples
training_pool = amazon_customers[
    ~amazon_customers["tweet_id"].isin(golden_ids)
].copy()

print("Total AmazonHelp customer messages:", len(amazon_customers))
print("Golden evaluation examples:", len(golden_df))
print("Remaining training pool:", len(training_pool))

NameError: name 'amazon_customers' is not defined

In [3]:
print("final_intent_df:", "final_intent_df" in globals())
print("golden_df:", "golden_df" in globals())
print("amazon_customers:", "amazon_customers" in globals())

final_intent_df: False
golden_df: False
amazon_customers: False


In [4]:
import pandas as pd

# Restore the 200 labeled examples
final_intent_df = pd.read_csv(
    "../data/processed/amazonhelp_intent_labeled.csv"
)

# Restore the golden evaluation set
golden_df = pd.read_csv(
    "../data/golden/amazonhelp_golden.csv"
)

print("Labeled dataset:", len(final_intent_df))
print("Golden set:", len(golden_df))

Labeled dataset: 200
Golden set: 200


In [5]:
import pandas as pd

final_intent_df = pd.read_csv(
    "../data/processed/amazonhelp_intent_labeled.csv"
)

golden_df = pd.read_csv(
    "../data/golden/amazonhelp_golden.csv"
)

print("Labeled dataset:", len(final_intent_df))
print("Golden set:", len(golden_df))

Labeled dataset: 200
Golden set: 200


In [6]:
import pandas as pd

csv_path = "../data/raw/twcs/twcs.csv"

# Get only AmazonHelp's tweets first
amazon_tweets = pd.read_csv(
    csv_path,
    usecols=[
        "tweet_id",
        "author_id",
        "inbound",
        "created_at",
        "text",
        "response_tweet_id",
        "in_response_to_tweet_id"
    ]
)

amazon_ids = set(
    amazon_tweets.loc[
        amazon_tweets["author_id"] == "AmazonHelp",
        "tweet_id"
    ]
)

# Customer tweets directly replying to AmazonHelp
amazon_customers = amazon_tweets[
    (amazon_tweets["inbound"] == True)
    & (
        amazon_tweets["in_response_to_tweet_id"]
        .isin(amazon_ids)
    )
].copy()

print("AmazonHelp tweets:", len(amazon_ids))
print("AmazonHelp customer messages:", len(amazon_customers))

AmazonHelp tweets: 169840
AmazonHelp customer messages: 100503


In [7]:
golden_ids = set(golden_df["tweet_id"])

training_pool = amazon_customers[
    ~amazon_customers["tweet_id"].isin(golden_ids)
].copy()

print("Total AmazonHelp customer messages:", len(amazon_customers))
print("Golden evaluation examples:", len(golden_df))
print("Remaining training pool:", len(training_pool))

Total AmazonHelp customer messages: 100503
Golden evaluation examples: 200
Remaining training pool: 100303


In [8]:
training_pool[["tweet_id", "text"]].head(10)

,tweet_id,text
182,270,@AmazonHelp ありがとうございます。\n今、電話で主人が対応していただいてます。
183,271,@AmazonHelp 電話で対応してもらいましたが改良されませんでした。\n保証期間も過ぎ...
185,274,@AmazonHelp こちらこそありがとうございました。
322,616,@AmazonHelp 3 different people have given 3 di...
324,619,@AmazonHelp I frankly don't have the patience ...
329,623,"@AmazonHelp Okay, danke für die Info"
333,627,@AmazonHelp @115826 Yeah this is crazy we’re l...
344,638,@AmazonHelp Hi ready for some help
347,641,@AmazonHelp Nothing there helped me with the E...
351,645,@AmazonHelp That page is useless - doesn’t all...


In [9]:
training_sample = training_pool.sample(
    n=1000,
    random_state=42
).reset_index(drop=True)

print("Training sample size:", len(training_sample))

Training sample size: 1000


In [10]:
# Look at 10 customer messages and the AmazonHelp tweet they replied to

conversation_check = amazon_customers[
    [
        "tweet_id",
        "text",
        "in_response_to_tweet_id"
    ]
].head(10)

conversation_check

,tweet_id,text,in_response_to_tweet_id
182,270,@AmazonHelp ありがとうございます。\n今、電話で主人が対応していただいてます。,269.0
183,271,@AmazonHelp 電話で対応してもらいましたが改良されませんでした。\n保証期間も過ぎ...,269.0
185,274,@AmazonHelp こちらこそありがとうございました。,273.0
322,616,@AmazonHelp 3 different people have given 3 di...,615.0
324,619,@AmazonHelp I frankly don't have the patience ...,618.0
329,623,"@AmazonHelp Okay, danke für die Info",622.0
333,627,@AmazonHelp @115826 Yeah this is crazy we’re l...,626.0
344,638,@AmazonHelp Hi ready for some help,639.0
347,641,@AmazonHelp Nothing there helped me with the E...,642.0
351,645,@AmazonHelp That page is useless - doesn’t all...,644.0


In [11]:
# Get the AmazonHelp responses for those 10 customer messages

amazon_lookup = amazon_tweets[
    ["tweet_id", "author_id", "text"]
].set_index("tweet_id")

for _, row in conversation_check.iterrows():
    parent_id = row["in_response_to_tweet_id"]

    print("=" * 80)
    print("CUSTOMER:")
    print(row["text"])

    print("\nAMAZONHELP RESPONSE:")
    print(amazon_lookup.loc[parent_id, "text"])

CUSTOMER:
@AmazonHelp ありがとうございます。
今、電話で主人が対応していただいてます。

AMAZONHELP RESPONSE:
@115770 こんにちは、アマゾン公式です。Fire TV Stickが見れないというのは、どのような状況でしょうか。一般的なトラブルシューティングを記載したヘルプがございますので、ご参照ください。https://t.co/2pbG55qJ7h ET
CUSTOMER:
@AmazonHelp 電話で対応してもらいましたが改良されませんでした。
保証期間も過ぎてるので買い直しになるんでしょうね。

AMAZONHELP RESPONSE:
@115770 こんにちは、アマゾン公式です。Fire TV Stickが見れないというのは、どのような状況でしょうか。一般的なトラブルシューティングを記載したヘルプがございますので、ご参照ください。https://t.co/2pbG55qJ7h ET
CUSTOMER:
@AmazonHelp こちらこそありがとうございました。

AMAZONHELP RESPONSE:
@115770 カスタマーサービスにてお問い合わせ済みとのことで、お手数をおかけいたしました。リプライいただきありがとうございました。ET
CUSTOMER:
@AmazonHelp 3 different people have given 3 different answers and I still don't have my order. Says delivered Saturday, was not, I was home all day

AMAZONHELP RESPONSE:
@115820 I'm sorry we've let you down! Without providing any personal information, will you describe the issue? We'd love to help. ^TN
CUSTOMER:
@AmazonHelp I frankly don't have the patience for another chat with your "customer service" people today.

AMAZONHELP

In [13]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [14]:
from src.conversations import build_customer_response_pairs

conversation_pairs = build_customer_response_pairs(
    amazon_customers,
    amazon_tweets
)

print("Customer-response pairs:", len(conversation_pairs))

print(
    conversation_pairs[
        ["customer_tweet_id", "customer_text", "response_text"]
    ].head(5)
)

Customer-response pairs: 100503
   customer_tweet_id                                      customer_text  \
0                270      @AmazonHelp ありがとうございます。\n今、電話で主人が対応していただいてます。   
1                271  @AmazonHelp 電話で対応してもらいましたが改良されませんでした。\n保証期間も過ぎ...   
2                274                      @AmazonHelp こちらこそありがとうございました。   
3                616  @AmazonHelp 3 different people have given 3 di...   
4                619  @AmazonHelp I frankly don't have the patience ...   

                                       response_text  
0  @115770 こんにちは、アマゾン公式です。Fire TV Stickが見れないというのは...  
1  @115770 こんにちは、アマゾン公式です。Fire TV Stickが見れないというのは...  
2  @115770 カスタマーサービスにてお問い合わせ済みとのことで、お手数をおかけいたしました...  
3  @115820 I'm sorry we've let you down! Without ...  
4  @115820 We'd like to take a further look into ...  


In [15]:
conversation_pairs.to_csv(
    "../data/processed/amazonhelp_conversation_pairs.csv",
    index=False
)

print(
    "Saved:",
    len(conversation_pairs),
    "conversation pairs"
)

Saved: 100503 conversation pairs


In [16]:
conversation_pairs["customer_text_clean"] = (
    conversation_pairs["customer_text"]
    .fillna("")
    .str.replace(r"@\w+", "", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

retrieval_pairs = conversation_pairs[
    conversation_pairs["customer_text_clean"].str.len() >= 20
].copy()

print("Original pairs:", len(conversation_pairs))
print("Retrieval pairs:", len(retrieval_pairs))
print(
    "Removed:",
    len(conversation_pairs) - len(retrieval_pairs)
)

Original pairs: 100503
Retrieval pairs: 90708
Removed: 9795


In [17]:
retrieval_pairs.to_csv(
    "../data/processed/amazonhelp_retrieval_pairs.csv",
    index=False
)

print(
    "Saved:",
    len(retrieval_pairs),
    "retrieval pairs"
)

Saved: 90708 retrieval pairs


In [18]:
import sys
!{sys.executable} -m pip install sentence-transformers faiss-cpu

     ---------------------------------------- 0.0/739.8 kB ? eta -:--:--
     --------------- ---------------------- 307.2/739.8 kB 9.6 MB/s eta 0:00:01
     --------------------------------- ---- 645.1/739.8 kB 8.1 MB/s eta 0:00:01
     -------------------------------------- 739.8/739.8 kB 6.7 MB/s eta 0:00:00
     ---------------------------------------- 0.0/16.2 MB ? eta -:--:--
     - -------------------------------------- 0.4/16.2 MB 8.7 MB/s eta 0:00:02
     - -------------------------------------- 0.7/16.2 MB 7.5 MB/s eta 0:00:03
     -- ------------------------------------- 1.0/16.2 MB 7.0 MB/s eta 0:00:03
     --- ------------------------------------ 1.3/16.2 MB 7.4 MB/s eta 0:00:03
     --- ------------------------------------ 1.6/16.2 MB 6.8 MB/s eta 0:00:03
     ---- ----------------------------------- 1.9/16.2 MB 6.8 MB/s eta 0:00:03
     ----- ---------------------------------- 2.2/16.2 MB 6.7 MB/s eta 0:00:03
     ------ --------------------------------- 2.5/16.2 MB 6.6 


[notice] A new release of pip is available: 23.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [19]:
from sentence_transformers import SentenceTransformer
import faiss

print("Sentence Transformers:", "OK")
print("FAISS:", "OK")

d:\Hiver_Assigenment\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Sentence Transformers: OK
FAISS: OK


In [20]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully")

d:\Hiver_Assigenment\.venv\lib\site-packages\huggingface_hub\file_download.py:142: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Admin\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4036.72it/s]


Embedding model loaded successfully


In [1]:
customer_texts = retrieval_pairs["customer_text_clean"].tolist()

embeddings = embedding_model.encode(
    customer_texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)

print("Embeddings shape:", embeddings.shape)

NameError: name 'retrieval_pairs' is not defined

In [2]:
import pandas as pd

retrieval_pairs = pd.read_csv(
    "../data/processed/amazonhelp_retrieval_pairs.csv"
)

print("Retrieval pairs:", len(retrieval_pairs))

Retrieval pairs: 90708


In [3]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully")

d:\Hiver_Assigenment\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2663.23it/s]


Embedding model loaded successfully


In [1]:
customer_texts = retrieval_pairs["customer_text_clean"].tolist()

embeddings = embedding_model.encode(
    customer_texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)

print("Embeddings shape:", embeddings.shape)

NameError: name 'retrieval_pairs' is not defined

In [2]:
import pandas as pd

retrieval_pairs = pd.read_csv(
    "../data/processed/amazonhelp_retrieval_pairs.csv"
)

print("Retrieval pairs:", len(retrieval_pairs))

Retrieval pairs: 90708


In [3]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully")

d:\Hiver_Assigenment\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3945.33it/s]


Embedding model loaded successfully


In [4]:
customer_texts = retrieval_pairs["customer_text_clean"].tolist()

embeddings = embedding_model.encode(
    customer_texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)

print("Embeddings shape:", embeddings.shape)

Batches: 100%|██████████| 1418/1418 [14:54<00:00,  1.59it/s]


Embeddings shape: (90708, 384)


In [5]:
print("Embeddings shape:", embeddings.shape)
print("Number of retrieval records:", len(retrieval_pairs))

Embeddings shape: (90708, 384)
Number of retrieval records: 90708


In [6]:
import faiss
import numpy as np

# Convert embeddings to float32 for FAISS
embedding_matrix = np.asarray(
    embeddings,
    dtype="float32"
)

# Create cosine-similarity index
index = faiss.IndexFlatIP(
    embedding_matrix.shape[1]
)

# Add historical customer-message embeddings
index.add(embedding_matrix)

print("FAISS index built successfully")
print("Indexed vectors:", index.ntotal)

FAISS index built successfully
Indexed vectors: 90708


In [7]:
def retrieve_similar_cases(query, top_k=5):
    """
    Find historically similar AmazonHelp customer messages.
    """

    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True
    )

    query_embedding = np.asarray(
        query_embedding,
        dtype="float32"
    )

    scores, indices = index.search(
        query_embedding,
        top_k
    )

    results = retrieval_pairs.iloc[indices[0]].copy()
    results["similarity_score"] = scores[0]

    return results

In [8]:
results = retrieve_similar_cases(
    "My package has not arrived yet",
    top_k=5
)

results[
    [
        "customer_text",
        "response_text",
        "similarity_score"
    ]
]

,customer_text,response_text,similarity_score
18937,@AmazonHelp My package still hasn’t arrived 😥,@218242 Delivery delays are possible but shoul...,0.836763
66087,@AmazonHelp Didn't received my package today,"@148218 parcel by 21:00 today, please let us k...",0.809228
5936,@AmazonHelp My package still hasn’t arrived. ...,@153437 Thank you for reaching out to us! Plea...,0.797952
15676,@AmazonHelp Still have yet to receive my package,@202513 I'm sorry for the wait! We deliver as ...,0.790921
7630,@AmazonHelp two of my packages haven’t even sh...,@164614 Have we missed the delivery date? If s...,0.782918


In [10]:
import sys
import os

project_root = os.path.abspath("..")

if project_root not in sys.path:
    sys.path.append(project_root)

print(project_root)

d:\Hiver_Assigenment


In [11]:
from src.retrieval import retrieve_similar_cases

print("Retrieval module imported successfully")

Retrieval module imported successfully


In [12]:
results = retrieve_similar_cases(
    query="My package has not arrived yet",
    embedding_model=embedding_model,
    index=index,
    retrieval_data=retrieval_pairs,
    top_k=5
)

results[
    [
        "customer_text",
        "response_text",
        "similarity_score"
    ]
]

,customer_text,response_text,similarity_score
18937,@AmazonHelp My package still hasn’t arrived 😥,@218242 Delivery delays are possible but shoul...,0.836763
66087,@AmazonHelp Didn't received my package today,"@148218 parcel by 21:00 today, please let us k...",0.809228
5936,@AmazonHelp My package still hasn’t arrived. ...,@153437 Thank you for reaching out to us! Plea...,0.797952
15676,@AmazonHelp Still have yet to receive my package,@202513 I'm sorry for the wait! We deliver as ...,0.790921
7630,@AmazonHelp two of my packages haven’t even sh...,@164614 Have we missed the delivery date? If s...,0.782918


In [13]:
import sys
!{sys.executable} -m pip install -U google-genai python-dotenv

     ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
     -- ------------------------------------- 0.1/1.1 MB 1.9 MB/s eta 0:00:01
     ------- -------------------------------- 0.2/1.1 MB 2.5 MB/s eta 0:00:01
     ------------ --------------------------- 0.3/1.1 MB 2.7 MB/s eta 0:00:01
     ------------------- -------------------- 0.5/1.1 MB 3.3 MB/s eta 0:00:01
     --------------------------- ------------ 0.7/1.1 MB 3.4 MB/s eta 0:00:01
     ------------------------------- -------- 0.9/1.1 MB 3.2 MB/s eta 0:00:01
     ---------------------------------------- 1.1/1.1 MB 3.4 MB/s eta 0:00:00
  Using cached tenacity-9.1.4-py3-none-any.whl (28 kB)
     ---------------------------------------- 0.0/472.6 kB ? eta -:--:--
     ------------------------- ------------ 317.4/472.6 kB 9.9 MB/s eta 0:00:01
     -------------------------------------- 472.6/472.6 kB 7.3 MB/s eta 0:00:00
  Using cached sniffio-1.3.1-py3-none-any.whl (10 kB)
  Using cached websockets-16.1.1-cp310-cp


[notice] A new release of pip is available: 23.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import sys
!{sys.executable} -m pip install -U google-genai python-dotenv

     ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
     ------------------ --------------------- 0.5/1.1 MB 10.7 MB/s eta 0:00:01
     ----------------------------- ---------- 0.8/1.1 MB 8.5 MB/s eta 0:00:01
     ---------------------------------------  1.1/1.1 MB 8.6 MB/s eta 0:00:01
     ---------------------------------------- 1.1/1.1 MB 7.7 MB/s eta 0:00:00
  Attempting uninstall: google-genai
    Found existing installation: google-genai 2.22.0
    Uninstalling google-genai-2.22.0:
      Successfully uninstalled google-genai-2.22.0



[notice] A new release of pip is available: 23.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.getenv("GEMINI_API_KEY")

print("API key loaded:", api_key is not None)

API key loaded: True


In [3]:
from google import genai
import os

client = genai.Client(
    api_key=os.getenv("GEMINI_API_KEY")
)

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Reply with exactly: Gemini API is working."
)

print(response.text)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Gemini API is working.


In [5]:
import sys
import os

project_root = os.path.abspath("..")

if project_root not in sys.path:
    sys.path.append(project_root)

print("Project root:", project_root)

Project root: d:\Hiver_Assigenment


In [6]:
from src.generation import generate_grounded_reply

print("Generation module imported successfully")

Generation module imported successfully


In [8]:
from src.retrieval import retrieve_similar_cases

print("Retrieval imported")

Retrieval imported


In [9]:
import pandas as pd

retrieval_pairs = pd.read_csv(
    "../data/processed/amazonhelp_retrieval_pairs.csv"
)

print("Retrieval pairs:", len(retrieval_pairs))

Retrieval pairs: 90708


In [10]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Embedding model loaded")

d:\Hiver_Assigenment\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1534.14it/s]


Embedding model loaded


In [11]:
import faiss
import numpy as np

customer_texts = retrieval_pairs["customer_text_clean"].tolist()

embeddings = embedding_model.encode(
    customer_texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)

embedding_matrix = np.asarray(
    embeddings,
    dtype="float32"
)

index = faiss.IndexFlatIP(
    embedding_matrix.shape[1]
)

index.add(embedding_matrix)

print("FAISS index:", index.ntotal)

Batches: 100%|██████████| 1418/1418 [22:20<00:00,  1.06it/s]


FAISS index: 90708


In [12]:
from dotenv import load_dotenv
from google import genai
import os

load_dotenv()

client = genai.Client(
    api_key=os.getenv("GEMINI_API_KEY")
)

print("Gemini client ready")

Gemini client ready


In [13]:
test_message = "My package has not arrived yet"

test_results = retrieve_similar_cases(
    query=test_message,
    embedding_model=embedding_model,
    index=index,
    retrieval_data=retrieval_pairs,
    top_k=5
)

reply = generate_grounded_reply(
    client=client,
    customer_message=test_message,
    intent="delivery_issue",
    retrieved_cases=test_results
)

print("CUSTOMER:")
print(test_message)

print("\nGENERATED REPLY:")
print(reply)

CUSTOMER:
My package has not arrived yet

GENERATED REPLY:
I'm sorry to hear your package hasn't arrived yet. Deliveries can happen as late as 8 PM. Please let us know if your order doesn't arrive by tomorrow, and we'll be happy to investigate further.


In [14]:
for i, row in test_results.iterrows():
    print("=" * 80)
    print("CUSTOMER CASE:")
    print(row["customer_text"])
    
    print("\nHISTORICAL RESPONSE:")
    print(row["response_text"])
    
    print("\nSIMILARITY:")
    print(round(row["similarity_score"], 3))

CUSTOMER CASE:
@AmazonHelp My package still hasn’t arrived 😥

HISTORICAL RESPONSE:
@218242 Delivery delays are possible but should be rare. Please let us know if your order doesn't arrive tomorrow! ^ST

SIMILARITY:
0.837
CUSTOMER CASE:
@AmazonHelp Didn't received my package today

HISTORICAL RESPONSE:
@148218 parcel by 21:00 today, please let us know so we can investigate further. ^HC (2/2)

SIMILARITY:
0.809
CUSTOMER CASE:
@AmazonHelp My package still hasn’t arrived.  Now what?

HISTORICAL RESPONSE:
@153437 Thank you for reaching out to us! Please let us know if your package does not arrive by Monday. We want to help! ^SM

SIMILARITY:
0.798
CUSTOMER CASE:
@AmazonHelp Still have yet to receive my package

HISTORICAL RESPONSE:
@202513 I'm sorry for the wait! We deliver as late as 8PM. Please keep us updated! ^WT

SIMILARITY:
0.791
CUSTOMER CASE:
@AmazonHelp two of my packages haven’t even shipped yet

HISTORICAL RESPONSE:
@164614 Have we missed the delivery date? If so, what does tracki

In [15]:
from src.escalation import decide_escalation

decision = decide_escalation(
    intent="delivery_issue",
    similarity_score=float(test_results["similarity_score"].iloc[0]),
    generated_reply=reply
)

print("Decision:", decision["decision"])
print("Reason:", decision["reason"])

Decision: auto
Reason: Intent is low-risk and relevant historical evidence was found.


In [16]:
from src.intents import IntentClassifier

intent_classifier = IntentClassifier()

intent_classifier.train(
    final_intent_df["text"],
    final_intent_df["final_intent"]
)

print("Intent classifier trained successfully")

NameError: name 'final_intent_df' is not defined

In [17]:
import pandas as pd

final_intent_df = pd.read_csv(
    "../data/processed/amazonhelp_intent_labeled.csv"
)

print("Labeled examples:", len(final_intent_df))
print("Missing labels:", final_intent_df["final_intent"].isna().sum())

Labeled examples: 200
Missing labels: 0


In [18]:
from src.intents import IntentClassifier

intent_classifier = IntentClassifier()

intent_classifier.train(
    final_intent_df["text"],
    final_intent_df["final_intent"]
)

print("Intent classifier trained successfully")

Intent classifier trained successfully


In [19]:
test_message = "My package has not arrived yet"

predicted_intent = intent_classifier.predict_one(
    test_message
)

print("Customer message:", test_message)
print("Predicted intent:", predicted_intent)

Customer message: My package has not arrived yet
Predicted intent: delivery_issue


In [20]:
from src.pipeline import run_support_agent

print("Pipeline imported successfully")

Pipeline imported successfully


In [27]:
result = run_support_agent(
    customer_message="My package has not arrived yet",
    intent_classifier=intent_classifier,
    embedding_model=embedding_model,
    index=index,
    retrieval_data=retrieval_pairs,
    gemini_client=client,
    top_k=5
)

print("CUSTOMER:")
print(result["customer_message"])

print("\nINTENT:")
print(result["intent"])

print("\nSIMILARITY SCORE:")
print(result["similarity_score"])

print("\nGENERATED REPLY:")
print(result["reply"])

print("\nDECISION:")
print(result["decision"])

print("\nREASON:")
print(result["reason"])

CUSTOMER:
My package has not arrived yet

INTENT:
delivery_issue

SIMILARITY SCORE:
0.8367633819580078

GENERATED REPLY:
I'm sorry to hear your package hasn't arrived. Please let us know if it doesn't arrive by the expected delivery date, and we'll investigate further.

DECISION:
auto

REASON:
Intent is low-risk and relevant historical evidence was found.


In [28]:
import os

os.makedirs("../data/processed", exist_ok=True)

print("Pipeline components ready.")
print("Golden set:", os.path.exists("../data/golden/amazonhelp_golden.csv"))
print("Retrieval pairs:", os.path.exists("../data/processed/amazonhelp_retrieval_pairs.csv"))

Pipeline components ready.
Golden set: True
Retrieval pairs: True


In [29]:
import os

print("Golden set exists:",
      os.path.exists("../data/golden/amazonhelp_golden.csv"))

print("AmazonHelp retrieval data exists:",
      os.path.exists("../data/processed/amazonhelp_retrieval_pairs.csv"))

print("Intent sample exists:",
      os.path.exists("../data/processed/amazonhelp_intent_sample.csv"))

Golden set exists: True
AmazonHelp retrieval data exists: True
Intent sample exists: True


In [30]:
import pandas as pd

golden_df = pd.read_csv(
    "../data/golden/amazonhelp_golden.csv"
)

print("Golden examples:", len(golden_df))
print(golden_df["intent"].value_counts())

Golden examples: 200


KeyError: 'intent'

In [31]:
print(golden_df.columns.tolist())
print("\nFirst 3 rows:")
display(golden_df.head(3))

['tweet_id', 'author_id', 'created_at', 'text', 'final_intent']

First 3 rows:


,tweet_id,author_id,created_at,text,final_intent
0,2470126,706559,Wed Nov 15 11:08:39 +0000 2017,@AmazonHelp Something was supposed to be deliv...,delivery_issue
1,1705395,516907,Mon Nov 06 22:45:42 +0000 2017,@AmazonHelp Thank you 🙌🏻,other
2,2957011,816249,Wed Nov 29 19:36:39 +0000 2017,@AmazonHelp C’est Amazon Logistics,other


In [32]:
print("Golden examples:", len(golden_df))
print(golden_df["final_intent"].value_counts())

Golden examples: 200
final_intent
delivery_issue             51
other                      49
complaint_or_escalation    34
order_status               20
refund                      8
payment_or_billing          8
product_availability        7
technical_support           7
prime_membership            6
return_or_cancellation      5
product_issue               5
Name: count, dtype: int64


In [33]:
# Load the full AmazonHelp customer dataset
amazon_customers = pd.read_csv(
    "../data/processed/amazonhelp_customer_messages.csv"
)

print("Total AmazonHelp customer messages:", len(amazon_customers))

FileNotFoundError: [Errno 2] No such file or directory: '../data/processed/amazonhelp_customer_messages.csv'

In [34]:
import os

processed_path = "../data/processed"

print("Files in data/processed:")
for file in os.listdir(processed_path):
    print(file)

Files in data/processed:
.gitkeep
amazonhelp_conversation_pairs.csv
amazonhelp_intent_discovery_labeled.csv
amazonhelp_intent_labeled.csv
amazonhelp_intent_labeling.csv
amazonhelp_intent_sample.csv
amazonhelp_retrieval_pairs.csv
amazonhelp_tweets.csv


In [35]:
amazonhelp_tweets = pd.read_csv(
    "../data/processed/amazonhelp_tweets.csv"
)

print("Rows:", len(amazonhelp_tweets))
print("\nColumns:")
print(amazonhelp_tweets.columns.tolist())

Rows: 422637

Columns:
['tweet_id', 'author_id', 'inbound', 'created_at', 'text', 'response_tweet_id', 'in_response_to_tweet_id']


In [36]:
print("amazon_customers exists:", "amazon_customers" in globals())

if "amazon_customers" in globals():
    print("Rows:", len(amazon_customers))
    print(amazon_customers["author_id"].value_counts().head())

amazon_customers exists: False


In [37]:
import pandas as pd
import os

raw_path = "../data/raw/twcs/twcs.csv"

# AmazonHelp's own tweets
amazon_tweet_ids = set()

for chunk in pd.read_csv(
    raw_path,
    usecols=["tweet_id", "author_id"],
    chunksize=100_000
):
    amazon_ids = chunk.loc[
        chunk["author_id"] == "AmazonHelp",
        "tweet_id"
    ]

    amazon_tweet_ids.update(amazon_ids.astype(str))

print("AmazonHelp tweet IDs:", len(amazon_tweet_ids))

AmazonHelp tweet IDs: 169840


In [38]:
customer_chunks = []

for chunk in pd.read_csv(
    raw_path,
    chunksize=100_000
):
    # Customer tweets
    inbound = chunk[chunk["inbound"] == True].copy()

    # Convert parent tweet ID to string for matching
    parent_ids = (
        pd.to_numeric(
            inbound["in_response_to_tweet_id"],
            errors="coerce"
        )
        .astype("Int64")
        .astype(str)
    )

    # Keep only customers directly replying to AmazonHelp
    mask = parent_ids.isin(amazon_tweet_ids)

    customer_chunks.append(inbound[mask])

amazon_customers = pd.concat(
    customer_chunks,
    ignore_index=True
)

print("Clean AmazonHelp customer messages:", len(amazon_customers))
print("Unique customer authors:", amazon_customers["author_id"].nunique())

Clean AmazonHelp customer messages: 100503
Unique customer authors: 40671


In [39]:
output_path = "../data/processed/amazonhelp_customer_messages.csv"

amazon_customers.to_csv(
    output_path,
    index=False
)

print("Saved:", output_path)

Saved: ../data/processed/amazonhelp_customer_messages.csv


In [40]:
golden_df = pd.read_csv(
    "../data/golden/amazonhelp_golden.csv"
)

golden_ids = set(
    golden_df["tweet_id"].astype(str)
)

amazon_customers["tweet_id"] = (
    amazon_customers["tweet_id"].astype(str)
)

overlap = amazon_customers["tweet_id"].isin(golden_ids).sum()

print("Golden examples:", len(golden_df))
print("Golden IDs found in customer dataset:", overlap)

Golden examples: 200
Golden IDs found in customer dataset: 200


In [41]:
# Remove all golden examples from the training pool
training_pool = amazon_customers[
    ~amazon_customers["tweet_id"].isin(golden_ids)
].copy()

print("Total clean customer messages:", len(amazon_customers))
print("Golden examples excluded:", len(amazon_customers) - len(training_pool))
print("Training pool:", len(training_pool))

Total clean customer messages: 100503
Golden examples excluded: 200
Training pool: 100303


In [42]:
training_sample = training_pool.sample(
    n=1000,
    random_state=42
).reset_index(drop=True)

print("Training samples:", len(training_sample))
display(training_sample[["tweet_id", "text"]].head(10))

Training samples: 1000


,tweet_id,text
0,315275,@AmazonHelp @115850 thanks my query has been r...
1,668492,@AmazonHelp Then why you claim of next day del...
2,312989,"@AmazonHelp Did while having a chat on FB, was..."
3,27898,@AmazonHelp Are you going to be offering anyth...
4,304638,@AmazonHelp Not helpful! I need the wrong ship...
5,2676823,"@AmazonHelp Yeah like I said, useless."
6,1689740,@AmazonHelp Associate told me that the item wa...
7,2383252,@AmazonHelp I couldn't contact due to your fau...
8,500811,@AmazonHelp Yes it came today
9,67532,@AmazonHelp That link is re reporting a suspic...


In [43]:
print("Training sample size:", len(training_sample))

print("\nMissing text:")
print(training_sample["text"].isna().sum())

print("\nEmpty text:")
print(
    (training_sample["text"].fillna("").str.strip() == "").sum()
)

Training sample size: 1000

Missing text:
0

Empty text:
0


In [44]:
label_test = training_sample.head(20).copy()

print("Test batch size:", len(label_test))

display(
    label_test[["tweet_id", "text"]]
)

Test batch size: 20


,tweet_id,text
0,315275,@AmazonHelp @115850 thanks my query has been r...
1,668492,@AmazonHelp Then why you claim of next day del...
2,312989,"@AmazonHelp Did while having a chat on FB, was..."
3,27898,@AmazonHelp Are you going to be offering anyth...
4,304638,@AmazonHelp Not helpful! I need the wrong ship...
5,2676823,"@AmazonHelp Yeah like I said, useless."
6,1689740,@AmazonHelp Associate told me that the item wa...
7,2383252,@AmazonHelp I couldn't contact due to your fau...
8,500811,@AmazonHelp Yes it came today
9,67532,@AmazonHelp That link is re reporting a suspic...


In [45]:
allowed_intents = [
    "delivery_issue",
    "order_status",
    "refund",
    "return_or_cancellation",
    "product_issue",
    "technical_support",
    "prime_membership",
    "payment_or_billing",
    "product_availability",
    "complaint_or_escalation",
    "other"
]

print("Number of intents:", len(allowed_intents))
print(allowed_intents)

Number of intents: 11
['delivery_issue', 'order_status', 'refund', 'return_or_cancellation', 'product_issue', 'technical_support', 'prime_membership', 'payment_or_billing', 'product_availability', 'complaint_or_escalation', 'other']


In [46]:
import json
import time


def label_messages_with_gemini(
    client,
    messages,
    allowed_intents
):
    """
    Assign one intent to each customer message using Gemini.
    """

    intent_list = ", ".join(allowed_intents)

    results = []

    for message in messages:

        prompt = f"""
You are labeling Amazon customer-support messages.

Choose exactly ONE intent from this list:

{intent_list}

Intent definitions:

- delivery_issue:
  Problems with delivery, delayed delivery, missing package,
  delivery date, shipping delay, or wrong delivery.

- order_status:
  Questions about the current status or progress of an order.

- refund:
  Requests or questions specifically about getting money back.

- return_or_cancellation:
  Returning an item or cancelling an order.

- product_issue:
  Problems with a physical product, damaged/defective product,
  or product not working as expected.

- technical_support:
  Technical problems with Amazon services, devices, apps,
  websites, login/access, or other technical functionality.

- prime_membership:
  Questions or problems specifically related to Amazon Prime.

- payment_or_billing:
  Payment, charge, billing, card, or account-payment problems.

- product_availability:
  Questions about whether a product is available, in stock,
  or when it will become available.

- complaint_or_escalation:
  Strong dissatisfaction, repeated unresolved issues,
  requests for escalation, or complaints about support.

- other:
  Messages that do not clearly fit the above categories,
  including simple acknowledgements or general questions.

Customer message:
{message}

Return ONLY valid JSON in this exact format:

{{"intent": "one_allowed_intent"}}
"""

        response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=prompt
        )

        raw = response.text.strip()

        try:
            parsed = json.loads(raw)
            intent = parsed["intent"]

            if intent not in allowed_intents:
                intent = "other"

        except (json.JSONDecodeError, KeyError):
            intent = "other"

        results.append(intent)

        # Small delay to reduce the chance of hitting rate limits
        time.sleep(0.5)

    return results

In [47]:
test_labels = label_messages_with_gemini(
    client=client,
    messages=label_test["text"].tolist(),
    allowed_intents=allowed_intents
)

label_test["pseudo_intent"] = test_labels

display(
    label_test[["tweet_id", "text", "pseudo_intent"]]
)

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-2.5-flash\nPlease retry in 31.22683409s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerMinutePerProjectPerModel-FreeTier', 'quotaDimensions': {'model': 'gemini-2.5-flash', 'location': 'global'}

In [48]:
training_50 = training_sample.head(50).copy()

display(
    training_50[["tweet_id", "text"]]
)

,tweet_id,text
0,315275,@AmazonHelp @115850 thanks my query has been r...
1,668492,@AmazonHelp Then why you claim of next day del...
2,312989,"@AmazonHelp Did while having a chat on FB, was..."
3,27898,@AmazonHelp Are you going to be offering anyth...
4,304638,@AmazonHelp Not helpful! I need the wrong ship...
5,2676823,"@AmazonHelp Yeah like I said, useless."
6,1689740,@AmazonHelp Associate told me that the item wa...
7,2383252,@AmazonHelp I couldn't contact due to your fau...
8,500811,@AmazonHelp Yes it came today
9,67532,@AmazonHelp That link is re reporting a suspic...


In [49]:
pd.set_option("display.max_colwidth", None)

for i, row in training_50.iterrows():
    print(f"\n--- {i} | Tweet ID: {row['tweet_id']} ---")
    print(row["text"])


--- 0 | Tweet ID: 315275 ---
@AmazonHelp @115850 thanks my query has been resolved.. 😀 Look forward to shop more

--- 1 | Tweet ID: 668492 ---
@AmazonHelp Then why you claim of next day delivery, stop fooling people @AmazonHelp @115850 This order was a next day delivery order, and now you request me to wait for the time frame and the irony is that you are hopeful

--- 2 | Tweet ID: 312989 ---
@AmazonHelp Did while having a chat on FB, was informed will get back, nothing happened

--- 3 | Tweet ID: 27898 ---
@AmazonHelp Are you going to be offering anything to your Puerto Rican customers for delayed/missing orders in the aftermath? Zero point in having prime

--- 4 | Tweet ID: 304638 ---
@AmazonHelp Not helpful! I need the wrong shipment to be picked ASAP, not dropping it anywhere. FIX IT!

--- 5 | Tweet ID: 2676823 ---
@AmazonHelp Yeah like I said, useless.

--- 6 | Tweet ID: 1689740 ---
@AmazonHelp Associate told me that the item wasn't @ a distribution center near me &amp; that I sh

In [50]:
for i in range(50):
    print("=" * 80)
    print(f"INDEX: {i}")
    print(f"TWEET ID: {training_50.iloc[i]['tweet_id']}")
    print(f"TEXT: {training_50.iloc[i]['text']}")
    print("=" * 80)

INDEX: 0
TWEET ID: 315275
TEXT: @AmazonHelp @115850 thanks my query has been resolved.. 😀 Look forward to shop more
INDEX: 1
TWEET ID: 668492
TEXT: @AmazonHelp Then why you claim of next day delivery, stop fooling people @AmazonHelp @115850 This order was a next day delivery order, and now you request me to wait for the time frame and the irony is that you are hopeful
INDEX: 2
TWEET ID: 312989
TEXT: @AmazonHelp Did while having a chat on FB, was informed will get back, nothing happened
INDEX: 3
TWEET ID: 27898
TEXT: @AmazonHelp Are you going to be offering anything to your Puerto Rican customers for delayed/missing orders in the aftermath? Zero point in having prime
INDEX: 4
TWEET ID: 304638
TEXT: @AmazonHelp Not helpful! I need the wrong shipment to be picked ASAP, not dropping it anywhere. FIX IT!
INDEX: 5
TWEET ID: 2676823
TEXT: @AmazonHelp Yeah like I said, useless.
INDEX: 6
TWEET ID: 1689740
TEXT: @AmazonHelp Associate told me that the item wasn't @ a distribution center near me &

In [51]:
for i in range(5, 10):
    print("=" * 80)
    print(f"INDEX: {i}")
    print(f"TWEET ID: {training_50.iloc[i]['tweet_id']}")
    print(f"TEXT: {training_50.iloc[i]['text']}")

INDEX: 5
TWEET ID: 2676823
TEXT: @AmazonHelp Yeah like I said, useless.
INDEX: 6
TWEET ID: 1689740
TEXT: @AmazonHelp Associate told me that the item wasn't @ a distribution center near me &amp; that I should know that Prime applies to date item is shipped.
INDEX: 7
TWEET ID: 2383252
TEXT: @AmazonHelp I couldn't contact due to your faulty IVRS, I only mailed them as you said, Today is my second day of wearing floaters @95892 chappals, all the staff members are staring at me and laughing behind my back, I can't bear the humiliation any more, Kindly arrange urgent replacemt
INDEX: 8
TWEET ID: 500811
TEXT: @AmazonHelp Yes it came today
INDEX: 9
TWEET ID: 67532
TEXT: @AmazonHelp That link is re reporting a suspicious email?


In [52]:
for i in range(10, 15):
    print("=" * 80)
    print(f"INDEX: {i}")
    print(f"TWEET ID: {training_50.iloc[i]['tweet_id']}")
    print(f"TEXT: {training_50.iloc[i]['text']}")

INDEX: 10
TWEET ID: 2402359
TEXT: @AmazonHelp No Till now..I got my product today n when I saw its completely different from wht I've ordered. Color n quality both r disappointing
INDEX: 11
TWEET ID: 1524027
TEXT: @AmazonHelp Still waiting. Not what I call next day delivery.
INDEX: 12
TWEET ID: 1007047
TEXT: @AmazonHelp Si un par de mensajes y sin respuesta. No quiero la devolución quiero mi pedido. Gracias
INDEX: 13
TWEET ID: 1646273
TEXT: @AmazonHelp The order says it was delivered yesterday. A refund was issued.
INDEX: 14
TWEET ID: 2931363
TEXT: @AmazonHelp Just a general question. When you call Amazon and request a U. S representatives you can’t get one because the offices are closed.


In [53]:
for i in range(15, 20):
    print("=" * 80)
    print(f"INDEX: {i}")
    print(f"TWEET ID: {training_50.iloc[i]['tweet_id']}")
    print(f"TEXT: {training_50.iloc[i]['text']}")

INDEX: 15
TWEET ID: 1760551
TEXT: @AmazonHelp I order from the uk so i guess its .co.uk
INDEX: 16
TWEET ID: 2367957
TEXT: @AmazonHelp It says it shipped today… a week after release. Next time ill go to a store to buy a preordered system. Geez… https://t.co/pwVtAeyeB3
INDEX: 17
TWEET ID: 1078202
TEXT: @AmazonHelp I've already shared the details contains Amazon invoice and Apple service centre feedback. Pls check and response asap.
INDEX: 18
TWEET ID: 1232102
TEXT: @AmazonHelp Is product was with me?
Why u sent my location after cancellation of 14 hrs?
You knew it I don't have any option for Amazon pay
#amazonpay https://t.co/S4pWaLbPpz
INDEX: 19
TWEET ID: 989115
TEXT: @AmazonHelp Amazon mistakenly cancelled my order. Contacted costumer ser &amp; they won't honour the original price. Please help. This is stressful.


In [54]:
for i in range(20, 25):
    print("=" * 80)
    print(f"INDEX: {i}")
    print(f"TWEET ID: {training_50.iloc[i]['tweet_id']}")
    print(f"TEXT: {training_50.iloc[i]['text']}")

INDEX: 20
TWEET ID: 833953
TEXT: @AmazonHelp Got this https://t.co/vO4VGU9Ksb
INDEX: 21
TWEET ID: 372163
TEXT: @AmazonHelp ok! mt obrigada :)
INDEX: 22
TWEET ID: 442966
TEXT: @AmazonHelp Nope been today since it was purchased on Friday
INDEX: 23
TWEET ID: 324446
TEXT: @AmazonHelp Please look at complaints registered for Order #407-4330002-1760315 @115850
INDEX: 24
TWEET ID: 2808969
TEXT: @AmazonHelp Info provided through the link. Please let me know the phone number that will call me. I do not answer unknown numbers.


In [55]:
for i in range(25, 30):
    print("=" * 80)
    print(f"INDEX: {i}")
    print(f"TWEET ID: {training_50.iloc[i]['tweet_id']}")
    print(f"TEXT: {training_50.iloc[i]['text']}")

INDEX: 25
TWEET ID: 2609953
TEXT: @AmazonHelp Done, pls look into this on priority. It's for a friend's wedding which is this coming weekend
INDEX: 26
TWEET ID: 742436
TEXT: @AmazonHelp merci beaucoup. Vous êtes le meilleur site d'e-commerce, et de loin.
INDEX: 27
TWEET ID: 1316007
TEXT: @AmazonHelp Literally the most useless response ever. Can I please direct message you?
INDEX: 28
TWEET ID: 1376744
TEXT: @AmazonHelp Supposedly the chat rep I spoke to already did. I've escalated before. Nothing happens.
INDEX: 29
TWEET ID: 1490151
TEXT: @AmazonHelp @AmazonHelp Also ist es jetzt möglich oder nich das ich mein geld für den Artikel wieder bekomme?


In [56]:
for i in range(30, 35):
    print("=" * 80)
    print(f"INDEX: {i}")
    print(f"TWEET ID: {training_50.iloc[i]['tweet_id']}")
    print(f"TEXT: {training_50.iloc[i]['text']}")

INDEX: 30
TWEET ID: 2893231
TEXT: @AmazonHelp Si già fatto
INDEX: 31
TWEET ID: 158792
TEXT: @AmazonHelp Thank you and I’ve figured it out. There are two listings for the show. One up front and that appears in searches that is 1080. One at the very back/end of the list and does not appear in searches that is UHD.
INDEX: 32
TWEET ID: 2396391
TEXT: @AmazonHelp I understand and thank you for prompt response!
INDEX: 33
TWEET ID: 1406807
TEXT: @AmazonHelp Thanks.
INDEX: 34
TWEET ID: 282796
TEXT: @AmazonHelp No, I got transferred to another agent instead, and they could not explain why it had not been transferred to manager. I have transcript, if you would like it.


In [57]:
for i in range(35, 40):
    print("=" * 80)
    print(f"INDEX: {i}")
    print(f"TWEET ID: {training_50.iloc[i]['tweet_id']}")
    print(f"TEXT: {training_50.iloc[i]['text']}")

INDEX: 35
TWEET ID: 314047
TEXT: @AmazonHelp Clearly couldn’t be bothered to send them so are now lying to me
INDEX: 36
TWEET ID: 26942
TEXT: @AmazonHelp Ah! Found @79347 on TuneIn. Shame it won’t work via @115888 but at least it now works on the Echo. Thanks for your help!
INDEX: 37
TWEET ID: 2697149
TEXT: @AmazonHelp Today would have been the day it should be delivered but the tracking date now says 20-22. Now i don’t have a swimsuit for my trip so THANKS FOR NOTHING
INDEX: 38
TWEET ID: 2772367
TEXT: @AmazonHelp Aaaaaaaaaaaaaa💙
INDEX: 39
TWEET ID: 2531368
TEXT: @AmazonHelp I don’t think so, I did order it then cancel one as I wasn’t 100%, then decided to order again. I’m guessing this is the confusion?


In [58]:
for i in range(40, 50):
    print("=" * 80)
    print(f"INDEX: {i}")
    print(f"TWEET ID: {training_50.iloc[i]['tweet_id']}")
    print(f"TEXT: {training_50.iloc[i]['text']}")

INDEX: 40
TWEET ID: 2373333
TEXT: @AmazonHelp It appears but no code
INDEX: 41
TWEET ID: 503518
TEXT: @AmazonHelp @AmazonHelp @115851 @115821  , kimdly reply
INDEX: 42
TWEET ID: 2338821
TEXT: @AmazonHelp I have ordered thousands of $$ worth of items on @115821 yet you have refused to refund or replace one worth about $35!!! Very wrong!!
INDEX: 43
TWEET ID: 2320755
TEXT: @AmazonHelp Payment by credit card for Prime no longer required already taken by you?
INDEX: 44
TWEET ID: 1325585
TEXT: @AmazonHelp @AmazonHelp and @115850 I have already left a review under customer service section - delivery and package option. Please follow it.
INDEX: 45
TWEET ID: 333202
TEXT: @AmazonHelp But it should be not more than mrp.actual price of mi a1 is 14999.
INDEX: 46
TWEET ID: 2721710
TEXT: @AmazonHelp Si no. No se escribiría..
INDEX: 47
TWEET ID: 382854
TEXT: @AmazonHelp Amazon. Because I can't opt out of Amazon delivery to let Ups or even USPS deliver. Amazon Fulfilment on the table.
INDEX: 48
TWEET I

In [59]:
for i in range(46, 50):
    print("=" * 80)
    print(f"INDEX: {i}")
    print(f"TWEET ID: {training_50.iloc[i]['tweet_id']}")
    print(f"TEXT: {training_50.iloc[i]['text']}")

INDEX: 46
TWEET ID: 2721710
TEXT: @AmazonHelp Si no. No se escribiría..
INDEX: 47
TWEET ID: 382854
TEXT: @AmazonHelp Amazon. Because I can't opt out of Amazon delivery to let Ups or even USPS deliver. Amazon Fulfilment on the table.
INDEX: 48
TWEET ID: 170999
TEXT: @AmazonHelp I’m sad to say it lead to few moments of false hope for me as it was exactly what I was for (the 43” model, anyway) but at least the price and model now match for all the other shoppers out there.
INDEX: 49
TWEET ID: 2584163
TEXT: @AmazonHelp  https://t.co/GauvEc9IgR


In [60]:
training_labels = [
    "other",
    "delivery_issue",
    "complaint_or_escalation",
    "delivery_issue",
    "return_or_cancellation",
    "complaint_or_escalation",
    "prime_membership",
    "product_issue",
    "other",
    "technical_support",
    "product_issue",
    "delivery_issue",
    "order_status",
    "refund",
    "other",
    "other",
    "delivery_issue",
    "product_issue",
    "payment_or_billing",
    "return_or_cancellation",
    "other",
    "other",
    "order_status",
    "complaint_or_escalation",
    "complaint_or_escalation",
    "complaint_or_escalation",
    "other",
    "complaint_or_escalation",
    "complaint_or_escalation",
    "refund",
    "other",
    "product_availability",
    "other",
    "other",
    "complaint_or_escalation",
    "complaint_or_escalation",
    "technical_support",
    "delivery_issue",
    "other",
    "return_or_cancellation",
    "technical_support",
    "complaint_or_escalation",
    "refund",
    "payment_or_billing",
    "delivery_issue",
    "payment_or_billing",
    "other",
    "delivery_issue",
    "product_availability",
    "other"
]

training_50["intent"] = training_labels

training_50.to_csv(
    "../data/processed/amazonhelp_training_50.csv",
    index=False
)

print("Training examples saved:", len(training_50))
print("\nIntent distribution:")
print(training_50["intent"].value_counts())

Training examples saved: 50

Intent distribution:
intent
other                      13
complaint_or_escalation    10
delivery_issue              7
return_or_cancellation      3
product_issue               3
technical_support           3
refund                      3
payment_or_billing          3
order_status                2
product_availability        2
prime_membership            1
Name: count, dtype: int64


In [61]:
training_100 = training_sample.iloc[50:100].copy().reset_index(drop=True)

print("Next training batch:", len(training_100))

Next training batch: 50


In [62]:
for i in range(0, 10):
    print("=" * 80)
    print(f"INDEX: {i + 50}")
    print(f"TWEET ID: {training_100.iloc[i]['tweet_id']}")
    print(f"TEXT: {training_100.iloc[i]['text']}")

INDEX: 50
TWEET ID: 1223548
TEXT: @AmazonHelp I don't understand how it can be estimated delivery Tuesday the 14th and the jump to the 20th for prime items.... 😤
INDEX: 51
TWEET ID: 434611
TEXT: @AmazonHelp DON'T BE AN IDIO IM SENDING MULTIPLE EMAILS WITH CONTACT DETAILS, CHECK BEFORE REVERTING LIKE AN IDIOTS
INDEX: 52
TWEET ID: 461317
TEXT: @AmazonHelp Amazon!
INDEX: 53
TWEET ID: 1568726
TEXT: @AmazonHelp I can't return the item because it was not delivered to me. I am not seeing a replace option.
INDEX: 54
TWEET ID: 1041757
TEXT: @AmazonHelp No, en ningún lado indica que se realizó con meses sin intereses.
Únicamente me hicieron el descuento del 10%
INDEX: 55
TWEET ID: 857764
TEXT: @AmazonHelp Device location, address are changed. I'm able to play from Saavn. But language is still in EN-US. EN-IN is not showing on the list.
INDEX: 56
TWEET ID: 2731115
TEXT: @AmazonHelp https://t.co/0pRFw0SAZY    audio between me and delivery person. @115850
INDEX: 57
TWEET ID: 1354501
TEXT: @AmazonHe

In [63]:
for i in range(56, 60):
    print("=" * 80)
    print(f"INDEX: {i}")
    print(f"TWEET ID: {training_sample.iloc[i]['tweet_id']}")
    print(f"TEXT: {training_sample.iloc[i]['text']}")

INDEX: 56
TWEET ID: 2731115
TEXT: @AmazonHelp https://t.co/0pRFw0SAZY    audio between me and delivery person. @115850
INDEX: 57
TWEET ID: 1354501
TEXT: @AmazonHelp  https://t.co/3kchUqFgQg
INDEX: 58
TWEET ID: 114408
TEXT: @AmazonHelp It's a Fire HD 3rd Gen.
INDEX: 59
TWEET ID: 1824383
TEXT: @AmazonHelp 返金してもらえるのですね！やってみます！


In [64]:
for i in range(60, 70):
    print("=" * 80)
    print(f"INDEX: {i}")
    print(f"TWEET ID: {training_sample.iloc[i]['tweet_id']}")
    print(f"TEXT: {training_sample.iloc[i]['text']}")

INDEX: 60
TWEET ID: 979056
TEXT: @AmazonHelp Required details have been shared with you. Plz check and do the needful
INDEX: 61
TWEET ID: 2004839
TEXT: @AmazonHelp Dankööö.
INDEX: 62
TWEET ID: 480335
TEXT: @AmazonHelp En caso de no ser así el que se fastidia soy yo ya que tendría que esperar hasta el lunes
INDEX: 63
TWEET ID: 1239303
TEXT: @AmazonHelp Your customer service is lacking also. 😒😒
INDEX: 64
TWEET ID: 246387
TEXT: @AmazonHelp @130687 Ninguna. De hecho, me pregunto si sería posible que en Amazon se previzualizara la compañía de transportes.  Con SEUR sólo tengo problemas.
INDEX: 65
TWEET ID: 2911658
TEXT: @AmazonHelp Will do. Thank you!
INDEX: 66
TWEET ID: 2206122
TEXT: @AmazonHelp Amazon is working on safari for me but still not good chrome.
INDEX: 67
TWEET ID: 1014259
TEXT: @AmazonHelp No information comes up on subtitles with that link https://t.co/UaS5PwRQob
INDEX: 68
TWEET ID: 877034
TEXT: @AmazonHelp Nope
It says in the Amazon website,the delivery guy came twice to deli

In [65]:
for i in range(66, 69):
    print("=" * 80)
    print(f"INDEX: {i}")
    print(f"TWEET ID: {training_sample.iloc[i]['tweet_id']}")
    print(f"TEXT: {training_sample.iloc[i]['text']}")

INDEX: 66
TWEET ID: 2206122
TEXT: @AmazonHelp Amazon is working on safari for me but still not good chrome.
INDEX: 67
TWEET ID: 1014259
TEXT: @AmazonHelp No information comes up on subtitles with that link https://t.co/UaS5PwRQob
INDEX: 68
TWEET ID: 877034
TEXT: @AmazonHelp Nope
It says in the Amazon website,the delivery guy came twice to deliver the pakage but nobody signed to collect the order
Connect by email


In [66]:
labels_60_69 = {
    60: "other",
    61: "other",
    62: "delivery_issue",
    63: "complaint_or_escalation",
    64: "delivery_issue",
    65: "other",
    66: "technical_support",
    67: "technical_support",
    68: "delivery_issue",
    69: "other"
}

In [67]:
for i, label in labels_60_69.items():
    training_100.loc[i - 50, "intent"] = label

In [68]:
for i in range(70, 75):
    print("=" * 80)
    print(f"INDEX: {i}")
    print(f"TWEET ID: {training_sample.iloc[i]['tweet_id']}")
    print(f"TEXT: {training_sample.iloc[i]['text']}")

INDEX: 70
TWEET ID: 277218
TEXT: @AmazonHelp @115821 it's now been now than 12 hours and still no response. Please advise on what to do next???!!!
INDEX: 71
TWEET ID: 1838972
TEXT: @AmazonHelp I'm unable to return the unopened, unused product. At least help me with that !
INDEX: 72
TWEET ID: 1972806
TEXT: @AmazonHelp Pas de soucis merci bien :)
INDEX: 73
TWEET ID: 1899905
TEXT: @AmazonHelp We can, but we have been filling in form since September. Is this ever going to be fixed?
INDEX: 74
TWEET ID: 2823925
TEXT: @AmazonHelp Yes. And paid for gift wrap.


In [69]:
labels_70_74 = {
    70: "complaint_or_escalation",
    71: "return_or_cancellation",
    72: "other",
    73: "technical_support",
    74: "payment_or_billing"
}

for i, label in labels_70_74.items():
    training_100.loc[i - 50, "intent"] = label

In [70]:
for i in range(75, 81):
    print("=" * 80)
    print(f"INDEX: {i}")
    print(f"TWEET ID: {training_sample.iloc[i]['tweet_id']}")
    print(f"TEXT: {training_sample.iloc[i]['text']}")

INDEX: 75
TWEET ID: 2343973
TEXT: @AmazonHelp Nothing inside package's, have attached pics of outside info https://t.co/Z45G9XEwr4
INDEX: 76
TWEET ID: 1953340
TEXT: @AmazonHelp issue is with seller support not buyer support ............@115851
INDEX: 77
TWEET ID: 111770
TEXT: @AmazonHelp Yes. Apparently it was signed for when no one was in and there’s a squiggle of a signature which isn’t mine. I asked for a call from amazon where the woman hung up on me and now I’m being told by a customer service chat that I might not get my parcel!
INDEX: 78
TWEET ID: 2826444
TEXT: @AmazonHelp Pas de souci !
INDEX: 79
TWEET ID: 1735446
TEXT: @AmazonHelp you need to start doing cs. 
This isn't  +ve CE
@132981 @209576 @118919 @4449 @524017 @54519 @54520
INDEX: 80
TWEET ID: 317258
TEXT: @AmazonHelp Still can’t get onto my account via the app though - any ideas?


In [71]:
labels_75_80 = {
    75: "product_issue",
    76: "other",
    77: "delivery_issue",
    78: "other",
    79: "complaint_or_escalation",
    80: "technical_support"
}

for i, label in labels_75_80.items():
    training_100.loc[i - 50, "intent"] = label

In [72]:
for i in range(81, 86):
    print("=" * 80)
    print(f"INDEX: {i}")
    print(f"TWEET ID: {training_sample.iloc[i]['tweet_id']}")
    print(f"TEXT: {training_sample.iloc[i]['text']}")

INDEX: 81
TWEET ID: 1735389
TEXT: @AmazonHelp Danke! Mit dem Kollegen telefoniert. Alles wird gut.
INDEX: 82
TWEET ID: 2064366
TEXT: @AmazonHelp  https://t.co/PqNv9QrRi6
INDEX: 83
TWEET ID: 1061325
TEXT: @AmazonHelp Why? Your customer service call was awful what are you going to do any differently?! Ruined my day and woke up my newborn! Disgrace
INDEX: 84
TWEET ID: 1229513
TEXT: @AmazonHelp Look at his hungry little face. https://t.co/KcvX5V2q9a
INDEX: 85
TWEET ID: 2633041
TEXT: @AmazonHelp Sim, comprei alguns ebooks. Mas não comprei nada no valor de 1 real.


In [73]:
labels_81_85 = {
    81: "other",
    82: "other",
    83: "complaint_or_escalation",
    84: "other",
    85: "payment_or_billing"
}

for i, label in labels_81_85.items():
    training_100.loc[i - 50, "intent"] = label

In [74]:
for i in range(86, 91):
    print("=" * 80)
    print(f"INDEX: {i}")
    print(f"TWEET ID: {training_sample.iloc[i]['tweet_id']}")
    print(f"TEXT: {training_sample.iloc[i]['text']}")

INDEX: 86
TWEET ID: 683097
TEXT: @AmazonHelp Whatever my order is - digital or not, I haven't received anything against my payment to you. Your siye mentioned doesn't help. Do needful or refund.
INDEX: 87
TWEET ID: 2378631
TEXT: @AmazonHelp Ma mettetela almeno in lingua originale
INDEX: 88
TWEET ID: 2474816
TEXT: @AmazonHelp 2/2 but that's not the point - you could have easily shipped a replacement. But you didn't. And this is now part of a pattern - several packages in the last few months. When I sent an email to designated customer service email ID I was told we'll give you $5 for your pains. Wow!
INDEX: 89
TWEET ID: 1090978
TEXT: @AmazonHelp i want to speak to cust care. No email, no chats, no tweets.
Why i m unable to reach to cust care person?
INDEX: 90
TWEET ID: 723662
TEXT: @AmazonHelp That would be AMZL US. didn't even stop.


In [75]:
labels_86_90 = {
    86: "refund",
    87: "other",
    88: "complaint_or_escalation",
    89: "complaint_or_escalation",
    90: "delivery_issue"
}

for i, label in labels_86_90.items():
    training_100.loc[i - 50, "intent"] = label

In [76]:
for i in range(91, 101):
    print("=" * 80)
    print(f"INDEX: {i}")
    print(f"TWEET ID: {training_sample.iloc[i]['tweet_id']}")
    print(f"TEXT: {training_sample.iloc[i]['text']}")

INDEX: 91
TWEET ID: 2944334
TEXT: @AmazonHelp Wasnt even a #BlackFriday or #CyberMonday item @115830 just failed ..... then made things worse
INDEX: 92
TWEET ID: 1731215
TEXT: @AmazonHelp I have amazon prime. I’ve got an email to confirm it will be delivered today. But I have just phoned CS who have said it will be here tomo.
INDEX: 93
TWEET ID: 2931312
TEXT: @AmazonHelp Oui, on m'a dit ce que je savais déjà : mon colis n'a pas pu être livré hier, et il doit être livré aujourd'hui.
INDEX: 94
TWEET ID: 2457191
TEXT: @AmazonHelp Go &amp; find it your own way!! I am tired of messaging and calling you. I had onces left you my email in private conversation. #Noprime #NoAmazon
INDEX: 95
TWEET ID: 1075195
TEXT: @AmazonHelp Can I reduce the size and print multiple on one page?
INDEX: 96
TWEET ID: 1379291
TEXT: @AmazonHelp Hi someone India call centre (on a Dublin no?) called but it was about something unrelated.  Had no idea about the case numbers from before.
INDEX: 97
TWEET ID: 38180
TEXT: @

In [77]:
labels_91_96 = {
    91: "product_issue",
    92: "delivery_issue",
    93: "delivery_issue",
    94: "complaint_or_escalation",
    95: "technical_support",
    96: "complaint_or_escalation"
}

for i, label in labels_91_96.items():
    training_100.loc[i - 50, "intent"] = label

In [78]:
for i in range(97, 101):
    print("=" * 80)
    print(f"INDEX: {i}")
    print(f"TWEET ID: {training_sample.iloc[i]['tweet_id']}")
    print(f"TEXT: {training_sample.iloc[i]['text']}")

INDEX: 97
TWEET ID: 38180
TEXT: @AmazonHelp Em 18/10 recebi um e-mail falando que a entrega ia ser adiantada, pois os livros chegariam antes do previsto (...)
INDEX: 98
TWEET ID: 2803307
TEXT: @AmazonHelp Unos pares de zapatos. Dios los bendiga a ustedes y a @107613 https://t.co/reLDmWx8X7
INDEX: 99
TWEET ID: 1215541
TEXT: @AmazonHelp Thanks I did that...received yet another email asking for my #Aadhaar !?! I mean..I don't..even..
INDEX: 100
TWEET ID: 2667943
TEXT: @AmazonHelp @115850 Received the correct order today. Thanks!


In [79]:
labels_97_100 = {
    97: "delivery_issue",
    98: "product_availability",
    99: "payment_or_billing",
    100: "order_status"
}

for i, label in labels_97_100.items():
    training_100.loc[i - 50, "intent"] = label

In [80]:
print("Total labels:", training_100["intent"].notna().sum())
print("Missing labels:", training_100["intent"].isna().sum())

print("\nIntent distribution:")
print(training_100["intent"].value_counts())

Total labels: 41
Missing labels: 10

Intent distribution:
intent
other                      11
delivery_issue              8
complaint_or_escalation     8
technical_support           5
payment_or_billing          3
product_issue               2
return_or_cancellation      1
refund                      1
product_availability        1
order_status                1
Name: count, dtype: int64


In [81]:
all_100_labels = {
    # 0-49
    0: "other",
    1: "delivery_issue",
    2: "complaint_or_escalation",
    3: "delivery_issue",
    4: "return_or_cancellation",
    5: "complaint_or_escalation",
    6: "prime_membership",
    7: "product_issue",
    8: "other",
    9: "technical_support",
    10: "product_issue",
    11: "delivery_issue",
    12: "order_status",
    13: "refund",
    14: "other",
    15: "other",
    16: "delivery_issue",
    17: "product_issue",
    18: "payment_or_billing",
    19: "return_or_cancellation",
    20: "other",
    21: "other",
    22: "order_status",
    23: "complaint_or_escalation",
    24: "complaint_or_escalation",
    25: "complaint_or_escalation",
    26: "other",
    27: "complaint_or_escalation",
    28: "complaint_or_escalation",
    29: "refund",
    30: "other",
    31: "product_availability",
    32: "other",
    33: "other",
    34: "complaint_or_escalation",
    35: "complaint_or_escalation",
    36: "technical_support",
    37: "delivery_issue",
    38: "other",
    39: "return_or_cancellation",
    40: "technical_support",
    41: "complaint_or_escalation",
    42: "refund",
    43: "payment_or_billing",
    44: "delivery_issue",
    45: "payment_or_billing",
    46: "other",
    47: "delivery_issue",
    48: "product_availability",
    49: "other",

    # 50-59
    50: "delivery_issue",
    51: "complaint_or_escalation",
    52: "other",
    53: "return_or_cancellation",
    54: "payment_or_billing",
    55: "technical_support",
    56: "complaint_or_escalation",
    57: "other",
    58: "product_issue",
    59: "refund",

    # 60-69
    60: "other",
    61: "other",
    62: "delivery_issue",
    63: "complaint_or_escalation",
    64: "delivery_issue",
    65: "other",
    66: "technical_support",
    67: "technical_support",
    68: "delivery_issue",
    69: "other",

    # 70-79
    70: "complaint_or_escalation",
    71: "return_or_cancellation",
    72: "other",
    73: "technical_support",
    74: "payment_or_billing",
    75: "product_issue",
    76: "other",
    77: "delivery_issue",
    78: "other",
    79: "complaint_or_escalation",

    # 80-89
    80: "technical_support",
    81: "other",
    82: "other",
    83: "complaint_or_escalation",
    84: "other",
    85: "payment_or_billing",
    86: "refund",
    87: "other",
    88: "complaint_or_escalation",
    89: "complaint_or_escalation",

    # 90-99
    90: "delivery_issue",
    91: "product_issue",
    92: "delivery_issue",
    93: "delivery_issue",
    94: "complaint_or_escalation",
    95: "technical_support",
    96: "complaint_or_escalation",
    97: "delivery_issue",
    98: "product_availability",
    99: "payment_or_billing",

    # 100 is outside the 100-example set
}

In [82]:
training_100 = training_sample.iloc[:100].copy()

training_100["intent"] = [
    all_100_labels[i]
    for i in range(100)
]

In [83]:
print("Total rows:", len(training_100))
print("Total labels:", training_100["intent"].notna().sum())
print("Missing labels:", training_100["intent"].isna().sum())

print("\nIntent distribution:")
print(training_100["intent"].value_counts())

Total rows: 100
Total labels: 100
Missing labels: 0

Intent distribution:
intent
other                      26
complaint_or_escalation    20
delivery_issue             16
technical_support           9
payment_or_billing          7
product_issue               6
return_or_cancellation      5
refund                      5
product_availability        3
order_status                2
prime_membership            1
Name: count, dtype: int64


In [84]:
training_100.to_csv(
    "../data/processed/amazonhelp_training_100.csv",
    index=False
)

print("Saved:", len(training_100), "training examples")

Saved: 100 training examples


In [85]:
import os

path = "../data/processed/amazonhelp_training_100.csv"

print("Exists:", os.path.exists(path))
print("Path:", os.path.abspath(path))

Exists: True
Path: d:\Hiver_Assigenment\data\processed\amazonhelp_training_100.csv


In [86]:
import pandas as pd

training_100 = pd.read_csv(
    "../data/processed/amazonhelp_training_100.csv"
)

golden = pd.read_csv(
    "../data/golden/amazonhelp_golden.csv"
)

print("Training:", len(training_100))
print("Golden:", len(golden))

Training: 100
Golden: 200


In [87]:
from src.intents import IntentClassifier

classifier = IntentClassifier()

classifier.train(
    training_100["text"],
    training_100["intent"]
)

print("Classifier trained successfully.")

Classifier trained successfully.


In [88]:
golden_predictions = classifier.predict(
    golden["text"]
)

golden["predicted_intent"] = golden_predictions

print(golden[[
    "text",
    "final_intent",
    "predicted_intent"
]].head(10))

                                                                                                                                                                     text  \
0  @AmazonHelp Something was supposed to be delivered today but it says on the order page ‘arrival at incorrect carrier facility’ and to ‘check back tuesday’ (yesterday)   
1                                                                                                                                                @AmazonHelp Thank you 🙌🏻   
2                                                                                                                                      @AmazonHelp C’est Amazon Logistics   
3                 @AmazonHelp yes I have been through all of the steps. I will check with nighbours further down the street tomorrow, but no notification of delivery 1/2   
4                                     @AmazonHelp But kindly help me to get it faster and help me to get it on 16 . I a k so know promi

In [89]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(
    golden["final_intent"],
    golden["predicted_intent"]
)

print(f"Golden accuracy: {accuracy:.2%}")

Golden accuracy: 38.00%


In [90]:
from sklearn.metrics import classification_report

print(
    classification_report(
        golden["final_intent"],
        golden["predicted_intent"],
        zero_division=0
    )
)

                         precision    recall  f1-score   support

complaint_or_escalation       0.32      0.56      0.40        34
         delivery_issue       0.71      0.20      0.31        51
           order_status       0.00      0.00      0.00        20
                  other       0.37      0.96      0.54        49
     payment_or_billing       0.00      0.00      0.00         8
       prime_membership       0.00      0.00      0.00         6
   product_availability       0.00      0.00      0.00         7
          product_issue       0.00      0.00      0.00         5
                 refund       0.00      0.00      0.00         8
 return_or_cancellation       0.00      0.00      0.00         5
      technical_support       0.00      0.00      0.00         7

               accuracy                           0.38       200
              macro avg       0.13      0.16      0.11       200
           weighted avg       0.33      0.38      0.28       200



In [91]:
from baselines.majority import majority_class_predict
from sklearn.metrics import accuracy_score

majority_predictions = majority_class_predict(
    training_100["intent"],
    len(golden)
)

majority_accuracy = accuracy_score(
    golden["final_intent"],
    majority_predictions
)

print(f"Majority baseline accuracy: {majority_accuracy:.2%}")

Majority baseline accuracy: 24.50%


In [92]:
from baselines.tfidf_logistic import train_tfidf_logistic, predict_tfidf_logistic
from sklearn.metrics import accuracy_score

vectorizer, model = train_tfidf_logistic(
    training_100["text"],
    training_100["intent"]
)

tfidf_predictions = predict_tfidf_logistic(
    vectorizer,
    model,
    golden["text"]
)

tfidf_accuracy = accuracy_score(
    golden["final_intent"],
    tfidf_predictions
)

print(f"TF-IDF + Logistic Regression accuracy: {tfidf_accuracy:.2%}")

TF-IDF + Logistic Regression accuracy: 38.00%


In [93]:
from sklearn.metrics import classification_report

print(
    classification_report(
        golden["final_intent"],
        tfidf_predictions,
        zero_division=0
    )
)

                         precision    recall  f1-score   support

complaint_or_escalation       0.32      0.56      0.40        34
         delivery_issue       0.71      0.20      0.31        51
           order_status       0.00      0.00      0.00        20
                  other       0.37      0.96      0.54        49
     payment_or_billing       0.00      0.00      0.00         8
       prime_membership       0.00      0.00      0.00         6
   product_availability       0.00      0.00      0.00         7
          product_issue       0.00      0.00      0.00         5
                 refund       0.00      0.00      0.00         8
 return_or_cancellation       0.00      0.00      0.00         5
      technical_support       0.00      0.00      0.00         7

               accuracy                           0.38       200
              macro avg       0.13      0.16      0.11       200
           weighted avg       0.33      0.38      0.28       200



In [94]:
from src.intents import IntentClassifier
from sklearn.metrics import accuracy_score, classification_report

classifier_balanced = IntentClassifier()

classifier_balanced.train(
    training_100["text"],
    training_100["intent"]
)

balanced_predictions = classifier_balanced.predict(
    golden["text"]
)

print(
    f"Balanced TF-IDF accuracy: "
    f"{accuracy_score(golden['final_intent'], balanced_predictions):.2%}"
)

print(
    classification_report(
        golden["final_intent"],
        balanced_predictions,
        zero_division=0
    )
)

Balanced TF-IDF accuracy: 38.00%
                         precision    recall  f1-score   support

complaint_or_escalation       0.32      0.56      0.40        34
         delivery_issue       0.71      0.20      0.31        51
           order_status       0.00      0.00      0.00        20
                  other       0.37      0.96      0.54        49
     payment_or_billing       0.00      0.00      0.00         8
       prime_membership       0.00      0.00      0.00         6
   product_availability       0.00      0.00      0.00         7
          product_issue       0.00      0.00      0.00         5
                 refund       0.00      0.00      0.00         8
 return_or_cancellation       0.00      0.00      0.00         5
      technical_support       0.00      0.00      0.00         7

               accuracy                           0.38       200
              macro avg       0.13      0.16      0.11       200
           weighted avg       0.33      0.38      0.28 

In [95]:
results = pd.DataFrame({
    "model": [
        "Majority baseline",
        "TF-IDF + Logistic Regression",
        "Balanced TF-IDF + Logistic Regression"
    ],
    "accuracy": [
        0.245,
        0.380,
        0.380
    ],
    "macro_f1": [
        None,
        0.11,
        0.11
    ]
})

results

,model,accuracy,macro_f1
0,Majority baseline,0.245,NaN
1,TF-IDF + Logistic Regression,0.380,0.11
2,Balanced TF-IDF + Logistic Regression,0.380,0.11


In [96]:
results.to_csv(
    "../results/classification_baselines.csv",
    index=False
)

print("Baseline results saved.")

Baseline results saved.


In [97]:
from src.semantic_intents import SemanticIntentClassifier

semantic_classifier = SemanticIntentClassifier(
    embedding_model
)

semantic_classifier.train(
    training_100["text"],
    training_100["intent"]
)

print("Semantic classifier trained.")

Semantic classifier trained.


In [98]:
test_message = "My package has not arrived yet"

prediction = semantic_classifier.predict_one(
    test_message
)

print("Message:", test_message)
print("Predicted intent:", prediction)

Message: My package has not arrived yet
Predicted intent: delivery_issue


In [99]:
semantic_predictions = semantic_classifier.predict(
    golden["text"],
    top_k=5
)

semantic_accuracy = accuracy_score(
    golden["final_intent"],
    semantic_predictions
)

print(f"Semantic classifier accuracy: {semantic_accuracy:.2%}")

Semantic classifier accuracy: 48.00%


In [100]:
print(
    classification_report(
        golden["final_intent"],
        semantic_predictions,
        zero_division=0
    )
)

                         precision    recall  f1-score   support

complaint_or_escalation       0.39      0.38      0.39        34
         delivery_issue       0.75      0.65      0.69        51
           order_status       1.00      0.05      0.10        20
                  other       0.42      0.92      0.57        49
     payment_or_billing       0.50      0.25      0.33         8
       prime_membership       0.00      0.00      0.00         6
   product_availability       0.00      0.00      0.00         7
          product_issue       0.00      0.00      0.00         5
                 refund       0.20      0.12      0.15         8
 return_or_cancellation       0.00      0.00      0.00         5
      technical_support       0.25      0.14      0.18         7

               accuracy                           0.48       200
              macro avg       0.32      0.23      0.22       200
           weighted avg       0.50      0.48      0.42       200



In [101]:
golden_analysis = golden.copy()

golden_analysis["predicted_intent"] = semantic_predictions

errors = golden_analysis[
    golden_analysis["final_intent"] != golden_analysis["predicted_intent"]
]

print("Total errors:", len(errors))

print(
    errors[
        ["text", "final_intent", "predicted_intent"]
    ].head(20).to_string(index=False)
)

Total errors: 104
                                                                                                                                                 text            final_intent        predicted_intent
                                                                                                       @AmazonHelp Sick of ur response.No need of it. complaint_or_escalation                   other
                                                                                               @AmazonHelp No resolution provided. Still facing issue complaint_or_escalation                   other
                                                                                                     @AmazonHelp What is the status still no response            order_status complaint_or_escalation
                                                                                                           @AmazonHelp I want it today else cancel it          delivery_issue                 

In [102]:
keywords = {
    "prime": ["prime"],
    "order": ["order", "ordered", "dispatch", "dispatched"],
    "refund": ["refund", "refunded", "money back"],
    "return": ["return", "returning", "returned"],
    "product": ["broken", "damaged", "defective", "wrong item"],
    "availability": ["available", "availability", "stock", "preorder"],
    "technical": ["app", "website", "login", "error", "link", "browser"]
}

for category, words in keywords.items():
    pattern = "|".join(words)

    matches = amazon_customers[
        amazon_customers["text"]
        .str.contains(pattern, case=False, na=False, regex=True)
    ]

    print(category, ":", len(matches))

prime : 4639
order : 10944
refund : 3605
return : 2019
product : 682
availability : 1659
technical : 9877


In [103]:
target_keywords = {
    "prime_membership": r"\bprime\b",
    "order_status": r"\border\b|\bordered\b|\bdispatch\b|\bdispatched\b",
    "refund": r"\brefund\b|\brefunded\b|\bmoney back\b",
    "return_or_cancellation": r"\breturn\b|\breturning\b|\breturned\b|\bcancel\b|\bcancellation\b",
    "product_issue": r"\bbroken\b|\bdamaged\b|\bdefective\b|\bwrong item\b",
    "product_availability": r"\bavailable\b|\bavailability\b|\bstock\b|\bpreorder\b",
    "technical_support": r"\bapp\b|\bwebsite\b|\blogin\b|\berror\b|\blink\b|\bbrowser\b"
}

target_candidates = {}

for intent, pattern in target_keywords.items():

    matches = amazon_customers[
        amazon_customers["text"]
        .str.contains(
            pattern,
            case=False,
            na=False,
            regex=True
        )
    ].copy()

    # Remove examples already in our 100-example training set
    matches = matches[
        ~matches["tweet_id"].isin(
            training_100["tweet_id"]
        )
    ]

    target_candidates[intent] = matches.sample(
        n=min(5, len(matches)),
        random_state=42
    ).reset_index(drop=True)

    print(
        intent,
        "->",
        len(target_candidates[intent]),
        "candidates"
    )

prime_membership -> 5 candidates
order_status -> 5 candidates
refund -> 5 candidates
return_or_cancellation -> 5 candidates
product_issue -> 5 candidates
product_availability -> 5 candidates
technical_support -> 5 candidates


In [104]:
for intent, df in target_candidates.items():

    print("\n" + "=" * 80)
    print("TARGET:", intent)

    for i, row in df.iterrows():
        print("-" * 80)
        print("INDEX:", i)
        print("TWEET ID:", row["tweet_id"])
        print("TEXT:", row["text"])


TARGET: prime_membership
--------------------------------------------------------------------------------
INDEX: 0
TWEET ID: 179445
TEXT: @AmazonHelp So, it never arrived, I've got my refund. Any chance of an answer to my original question: Why do you use Hermes for Prime deliveries?
--------------------------------------------------------------------------------
INDEX: 1
TWEET ID: 1049361
TEXT: @AmazonHelp Que se entrega hoy. Lo que molesta es pagar PRIME para entrega 1 día, lo compre el Miércoles 18 a las 21.00h y no ha llegado.
--------------------------------------------------------------------------------
INDEX: 2
TWEET ID: 133787
TEXT: @AmazonHelp She said fulfilled by Amazon prime
--------------------------------------------------------------------------------
INDEX: 3
TWEET ID: 155255
TEXT: @AmazonHelp Dice Amazon prime memb, por $212 pero yo la pagué por un año y caduca el 8 de junio. En serv a cliente me dijeron que debe ser que pedí un producto doble, pero no coinciden las 

In [105]:
prime_labels = {
    0: "prime_membership",
    1: "delivery_issue",
    2: "prime_membership",
    3: "payment_or_billing",
    4: "delivery_issue"
}

In [106]:
print("=" * 80)
print("TARGET: order_status")

for i, row in target_candidates["order_status"].iterrows():
    print("-" * 80)
    print("INDEX:", i)
    print("TWEET ID:", row["tweet_id"])
    print("TEXT:", row["text"])

TARGET: order_status
--------------------------------------------------------------------------------
INDEX: 0
TWEET ID: 1257467
TEXT: @AmazonHelp It’s been dispatched but it still doesn’t explain why my order was prioritised last
--------------------------------------------------------------------------------
INDEX: 1
TWEET ID: 836478
TEXT: @AmazonHelp Ordered a phone from #amazonprime.delivery was urgent as it was someone's gift. Failed to deliver. And delivery guy marks it delivered
--------------------------------------------------------------------------------
INDEX: 2
TWEET ID: 2791952
TEXT: @AmazonHelp There is no order. There is absolutely no reason for them to call me. I’ve said I didn’t ask for call back but so far in the last 30 minutes they called me 10 times
--------------------------------------------------------------------------------
INDEX: 3
TWEET ID: 2236236
TEXT: @AmazonHelp Just “Not yet dispatched. We'll e-mail you when we have a delivery date”
-------------------

In [107]:
order_status_labels = {
    0: "order_status",
    1: "delivery_issue",
    2: "complaint_or_escalation",
    3: "order_status",
    4: "delivery_issue"
}

In [108]:
print("=" * 80)
print("TARGET: refund")

for i, row in target_candidates["refund"].iterrows():
    print("-" * 80)
    print("INDEX:", i)
    print("TWEET ID:", row["tweet_id"])
    print("TEXT:", row["text"])

TARGET: refund
--------------------------------------------------------------------------------
INDEX: 0
TWEET ID: 1941523
TEXT: @AmazonHelp I demanded a refund for one because I hadn’t received the item and I’m traveling so I can’t get it before I leave. I reordered
--------------------------------------------------------------------------------
INDEX: 1
TWEET ID: 215207
TEXT: @AmazonHelp Spoke to rep and she can only replace 3 parts and refund the other parts. Disappointing because I was told it required a signature.
--------------------------------------------------------------------------------
INDEX: 2
TWEET ID: 755037
TEXT: @AmazonHelp No and I'm not even going to, I don't even want the product anymore. I'll just request a refund/return if it ever does arrive.
--------------------------------------------------------------------------------
INDEX: 3
TWEET ID: 897580
TEXT: @AmazonHelp Fyi: UPS finally picked up today. Now I need to process a refund for shipping from Amazon.
-------

In [109]:
refund_labels = {
    0: "refund",
    1: "refund",
    2: "refund",
    3: "refund",
    4: "refund"
}

In [110]:
print("=" * 80)
print("TARGET: return_or_cancellation")

for i, row in target_candidates["return_or_cancellation"].iterrows():
    print("-" * 80)
    print("INDEX:", i)
    print("TWEET ID:", row["tweet_id"])
    print("TEXT:", row["text"])

TARGET: return_or_cancellation
--------------------------------------------------------------------------------
INDEX: 0
TWEET ID: 1240427
TEXT: @AmazonHelp I return an order..your guy collected it today but didn't give any acknowledgement receipt..is it fine??
--------------------------------------------------------------------------------
INDEX: 1
TWEET ID: 1615579
TEXT: @AmazonHelp It's more than 12 hrs nd u hv not resolved my issue! You all are frauds and criminals! Why are not returning my money?
--------------------------------------------------------------------------------
INDEX: 2
TWEET ID: 2394441
TEXT: @AmazonHelp I contacted your customer service team. Even they are unable to cancel the order.
--------------------------------------------------------------------------------
INDEX: 3
TWEET ID: 461296
TEXT: @AmazonHelp UPS has no idea where this package is. I need your help to cancel this order as soon as possible.
--------------------------------------------------------------

In [111]:
return_labels = {
    0: "return_or_cancellation",
    1: "refund",
    2: "return_or_cancellation",
    3: "return_or_cancellation",
    4: "return_or_cancellation"
}

In [112]:
print("=" * 80)
print("TARGET: product_issue")

for i, row in target_candidates["product_issue"].iterrows():
    print("-" * 80)
    print("INDEX:", i)
    print("TWEET ID:", row["tweet_id"])
    print("TEXT:", row["text"])

TARGET: product_issue
--------------------------------------------------------------------------------
INDEX: 0
TWEET ID: 2927048
TEXT: @AmazonHelp Apparently it’s coming tomorrow now. So why do I pay so much for prime if my packages are damaged and constantly delayed. Ups and that other random carrier are the worst. Or things just never come and I get no support from the call line so I don’t bother.
--------------------------------------------------------------------------------
INDEX: 1
TWEET ID: 836164
TEXT: @AmazonHelp you support team is helpful. The technical support team is busy and I got defective mobile. Amazon is not bothered get refund request.
--------------------------------------------------------------------------------
INDEX: 2
TWEET ID: 2781951
TEXT: @AmazonHelp Looking to hit "Login &amp; security" page, and "Sign In Using Alternative Factors of Authentication" is sending me to broken link https://t.co/3yS91cUz6M.  Support thinks I should get an email when encounterin

In [113]:
product_issue_labels = {
    0: "delivery_issue",
    1: "product_issue",
    2: "technical_support",
    3: "product_issue",
    4: "product_issue"
}

In [115]:
print("=" * 80)
print("TARGET: product_availability")

for i, row in target_candidates["product_availability"].iterrows():
    print("-" * 80)
    print("INDEX:", i)
    print("TWEET ID:", row["tweet_id"])
    print("TEXT:", row["text"])

TARGET: product_availability
--------------------------------------------------------------------------------
INDEX: 0
TWEET ID: 191778
TEXT: @AmazonHelp @161066 you are blantantly lying just as ea did if we didnt preorder before oct 1st we dont get early beta so i hope someone sues the shit out of u
--------------------------------------------------------------------------------
INDEX: 1
TWEET ID: 1108270
TEXT: @AmazonHelp Sí, hace una semana, aunque entonces también estaba temporalmente sin stock
--------------------------------------------------------------------------------
INDEX: 2
TWEET ID: 954298
TEXT: @AmazonHelp @341932 Amazing, no wonder your stock is near all-time highs
--------------------------------------------------------------------------------
INDEX: 3
TWEET ID: 96711
TEXT: @AmazonHelp But I don't want a refund, I want a replacement. Is that not possible? I don't see that option. The TV doesn't seem to be available right now, and even if it was it wouldn't be the same 

In [116]:
availability_labels = {
    0: "product_availability",
    1: "product_availability",
    2: "other",
    3: "product_availability",
    4: "product_availability"
}

In [117]:
print("=" * 80)
print("TARGET: technical_support")

for i, row in target_candidates["technical_support"].iterrows():
    print("-" * 80)
    print("INDEX:", i)
    print("TWEET ID:", row["tweet_id"])
    print("TEXT:", row["text"])

TARGET: technical_support
--------------------------------------------------------------------------------
INDEX: 0
TWEET ID: 2801410
TEXT: @AmazonHelp I already sent an email thourgh your app. Please, I need this package!
--------------------------------------------------------------------------------
INDEX: 1
TWEET ID: 1320490
TEXT: @AmazonHelp Hola buenas, hable con una chica de atención al cliente que me envió un link a mi correo para solucionarlo y pode reclamar pero no puedo acceder a tal correo. ¿Qué puedo hacer? Gracias
--------------------------------------------------------------------------------
INDEX: 2
TWEET ID: 179204
TEXT: @AmazonHelp I used the link in your tweet - lead me to a log in page - I CANT LOG IN - is that loud enough?
--------------------------------------------------------------------------------
INDEX: 3
TWEET ID: 461277
TEXT: @AmazonHelp Now in your app, just noticed a little inconsistency. Scratch post says now expected later, I tap on it, it says arrivin

In [118]:
technical_labels = {
    0: "delivery_issue",
    1: "technical_support",
    2: "technical_support",
    3: "technical_support",
    4: "complaint_or_escalation"
}

In [119]:
print("Current training examples:", len(training_100))

new_labels = {}

new_labels.update(prime_labels)
new_labels.update(order_status_labels)
new_labels.update(refund_labels)
new_labels.update(return_labels)
new_labels.update(product_issue_labels)
new_labels.update(availability_labels)
new_labels.update(technical_labels)

print("New candidate labels collected:", len(new_labels))

Current training examples: 100
New candidate labels collected: 5


In [120]:
new_labels = {}

# Prime candidates
for i, label in prime_labels.items():
    tweet_id = target_candidates["prime_membership"].iloc[i]["tweet_id"]
    new_labels[tweet_id] = label

# Order status candidates
for i, label in order_status_labels.items():
    tweet_id = target_candidates["order_status"].iloc[i]["tweet_id"]
    new_labels[tweet_id] = label

# Refund candidates
for i, label in refund_labels.items():
    tweet_id = target_candidates["refund"].iloc[i]["tweet_id"]
    new_labels[tweet_id] = label

# Return/cancellation candidates
for i, label in return_labels.items():
    tweet_id = target_candidates["return_or_cancellation"].iloc[i]["tweet_id"]
    new_labels[tweet_id] = label

# Product issue candidates
for i, label in product_issue_labels.items():
    tweet_id = target_candidates["product_issue"].iloc[i]["tweet_id"]
    new_labels[tweet_id] = label

# Product availability candidates
for i, label in availability_labels.items():
    tweet_id = target_candidates["product_availability"].iloc[i]["tweet_id"]
    new_labels[tweet_id] = label

# Technical support candidates
for i, label in technical_labels.items():
    tweet_id = target_candidates["technical_support"].iloc[i]["tweet_id"]
    new_labels[tweet_id] = label

print("Unique new labeled examples:", len(new_labels))

Unique new labeled examples: 35


In [121]:
existing_ids = set(training_100["tweet_id"])

duplicate_ids = set(new_labels.keys()) & existing_ids

print("Existing training examples:", len(existing_ids))
print("New labeled examples:", len(new_labels))
print("Duplicates:", len(duplicate_ids))

Existing training examples: 100
New labeled examples: 35
Duplicates: 0


In [122]:
new_training = amazon_customers[
    amazon_customers["tweet_id"].isin(new_labels.keys())
].copy()

new_training["intent"] = new_training["tweet_id"].map(new_labels)

training_135 = pd.concat(
    [training_100, new_training],
    ignore_index=True
)

print("Total training examples:", len(training_135))
print("\nIntent distribution:")
print(training_135["intent"].value_counts())

Total training examples: 135

Intent distribution:
intent
other                      27
delivery_issue             22
complaint_or_escalation    22
technical_support          13
refund                     11
product_issue               9
return_or_cancellation      9
payment_or_billing          8
product_availability        7
order_status                4
prime_membership            3
Name: count, dtype: int64


In [123]:
training_135.to_csv(
    "../data/processed/amazonhelp_training_135.csv",
    index=False
)

print("Saved successfully.")

Saved successfully.


In [124]:
semantic_classifier_135 = SemanticIntentClassifier(
    embedding_model
)

semantic_classifier_135.train(
    training_135["text"],
    training_135["intent"]
)

print("Semantic classifier trained on 135 examples.")

Semantic classifier trained on 135 examples.


In [125]:
semantic_predictions_135 = semantic_classifier_135.predict(
    golden["text"],
    top_k=5
)

semantic_accuracy_135 = accuracy_score(
    golden["final_intent"],
    semantic_predictions_135
)

print(
    f"Semantic classifier (135) accuracy: "
    f"{semantic_accuracy_135:.2%}"
)

Semantic classifier (135) accuracy: 49.00%


In [126]:
print(
    classification_report(
        golden["final_intent"],
        semantic_predictions_135,
        zero_division=0
    )
)

                         precision    recall  f1-score   support

complaint_or_escalation       0.39      0.38      0.39        34
         delivery_issue       0.82      0.55      0.66        51
           order_status       0.62      0.25      0.36        20
                  other       0.45      0.92      0.60        49
     payment_or_billing       0.50      0.25      0.33         8
       prime_membership       0.00      0.00      0.00         6
   product_availability       0.00      0.00      0.00         7
          product_issue       0.00      0.00      0.00         5
                 refund       0.36      0.50      0.42         8
 return_or_cancellation       0.00      0.00      0.00         5
      technical_support       0.25      0.14      0.18         7

               accuracy                           0.49       200
              macro avg       0.31      0.27      0.27       200
           weighted avg       0.49      0.49      0.45       200



In [127]:
results.loc[len(results)] = [
    "Semantic KNN (135 labels)",
    0.49,
    0.27
]

results

,model,accuracy,macro_f1
0,Majority baseline,0.245,NaN
1,TF-IDF + Logistic Regression,0.380,0.11
2,Balanced TF-IDF + Logistic Regression,0.380,0.11
3,Semantic KNN (135 labels),0.490,0.27


In [128]:
results.to_csv(
    "../results/classification_results.csv",
    index=False
)

In [130]:
for query in test_queries:
    print("=" * 80)
    print("QUERY:", query)

    retrieved = retrieve_similar_cases(
        query,
        embedding_model,
        index,
        retrieval_pairs,
        top_k=3
    )

    display(
        retrieved[
            [
                "customer_text",
                "response_text",
                "similarity_score"
            ]
        ]
    )

QUERY: My package has not arrived yet


,customer_text,response_text,similarity_score
18937,@AmazonHelp My package still hasn’t arrived 😥,@218242 Delivery delays are possible but should be rare. Please let us know if your order doesn't arrive tomorrow! ^ST,0.836763
66087,@AmazonHelp Didn't received my package today,"@148218 parcel by 21:00 today, please let us know so we can investigate further. ^HC (2/2)",0.809228
5936,@AmazonHelp My package still hasn’t arrived. Now what?,@153437 Thank you for reaching out to us! Please let us know if your package does not arrive by Monday. We want to help! ^SM,0.797952


QUERY: I want a refund for my order


,customer_text,response_text,similarity_score
57140,@AmazonHelp Please cancel my order I want a full refund,@169841 I'm sorry for the frustration. We'd like to look into this with you here: https://t.co/CYqkdUakWJ ^SJ,0.849793
64344,@AmazonHelp I just want a refund,@583077 I'm so sorry for the difficulty you're having. Feel free to call us anytime using this number: 1-888-280-4331 ^MH,0.833764
77301,@AmazonHelp How do u get a refund please ?,"@720882 Hi, sorry to hear that, you can find instructions and info about canceling orders here: https://t.co/kAxwnwtxLM. ^JJ",0.814792


QUERY: My Amazon order is damaged


,customer_text,response_text,similarity_score
57607,@AmazonHelp My order is loss who is responsible amazon.in,@491795 Please don’t provide order details as we consider them to be personal info. Our Twitter page is visible to public. (2/2)^BS,0.770851
17083,@AmazonHelp I ordered from Amazon. Amazon needs to fix this?,@208873 Have you had a chance to contact USPS directly for additional insight? Their info is listed here: https://t.co/S8k5E5RiGH ^WM,0.749875
68451,"@AmazonHelp Package damaged in transit and not delivered, told to contact amazon CS.. no offer to send out order again (yes it's all in stock) and no apology.. was only offered a refund which will take 5-7 days to reach my bank account.","@656383 This definitely isn't what we strive for! Without providing personal account information, could you tell us more about what's going on? We'd love to help in any way we can! ^ML",0.739637


QUERY: I cannot log into my account


,customer_text,response_text,similarity_score
40843,@AmazonHelp I cannot login into my account .com,@375313 Hi Adriana! You've come to the right place. How can we be of service today? ^SY,0.896963
17158,@AmazonHelp I am unable to login,"@182608 Hello, so sorry to hear this. Could you kindly elaborate the issue, so that we can assist you with this.",0.865127
11256,@AmazonHelp I am not able to login,@182868 Sorry for the inconvenience. Contact our support team here: https://t.co/vlvfJr4nN9 we will assist you further. ^GU,0.853878


QUERY: I want to cancel my order


,customer_text,response_text,similarity_score
62188,@AmazonHelp This really doesn't help. I simply want to cancel an order.,@531655 I understand your concern! Please reach out to us via the link here: https://t.co/oGDklGgbCZ. We want to help! ^GM,0.883387
35493,@AmazonHelp I'm trying to cancel the order.,@322585 Sorry to hear this! We'll have your order out to you as soon as we can! ^RL,0.882940
7343,@AmazonHelp I hve done all these steps but still i am not able to cancel my order,@161170 We'd be happy to help! These steps will help walk you through canceling the order: https://t.co/vmXqEMJN6J ^ML,0.881764


In [131]:
test_queries = [
    "My package has not arrived yet",
    "I want a refund for my order",
    "My Amazon order is damaged",
    "I cannot log into my account"
]

for query in test_queries:
    print("=" * 80)
    print("QUERY:", query)

    retrieved = retrieve_similar_cases(
        query,
        embedding_model,
        index,
        retrieval_pairs,
        top_k=3
    )

    display(
        retrieved[
            [
                "customer_text",
                "response_text",
                "similarity_score"
            ]
        ]
    )

QUERY: My package has not arrived yet


,customer_text,response_text,similarity_score
18937,@AmazonHelp My package still hasn’t arrived 😥,@218242 Delivery delays are possible but should be rare. Please let us know if your order doesn't arrive tomorrow! ^ST,0.836763
66087,@AmazonHelp Didn't received my package today,"@148218 parcel by 21:00 today, please let us know so we can investigate further. ^HC (2/2)",0.809228
5936,@AmazonHelp My package still hasn’t arrived. Now what?,@153437 Thank you for reaching out to us! Please let us know if your package does not arrive by Monday. We want to help! ^SM,0.797952


QUERY: I want a refund for my order


,customer_text,response_text,similarity_score
57140,@AmazonHelp Please cancel my order I want a full refund,@169841 I'm sorry for the frustration. We'd like to look into this with you here: https://t.co/CYqkdUakWJ ^SJ,0.849793
64344,@AmazonHelp I just want a refund,@583077 I'm so sorry for the difficulty you're having. Feel free to call us anytime using this number: 1-888-280-4331 ^MH,0.833764
77301,@AmazonHelp How do u get a refund please ?,"@720882 Hi, sorry to hear that, you can find instructions and info about canceling orders here: https://t.co/kAxwnwtxLM. ^JJ",0.814792


QUERY: My Amazon order is damaged


,customer_text,response_text,similarity_score
57607,@AmazonHelp My order is loss who is responsible amazon.in,@491795 Please don’t provide order details as we consider them to be personal info. Our Twitter page is visible to public. (2/2)^BS,0.770851
17083,@AmazonHelp I ordered from Amazon. Amazon needs to fix this?,@208873 Have you had a chance to contact USPS directly for additional insight? Their info is listed here: https://t.co/S8k5E5RiGH ^WM,0.749875
68451,"@AmazonHelp Package damaged in transit and not delivered, told to contact amazon CS.. no offer to send out order again (yes it's all in stock) and no apology.. was only offered a refund which will take 5-7 days to reach my bank account.","@656383 This definitely isn't what we strive for! Without providing personal account information, could you tell us more about what's going on? We'd love to help in any way we can! ^ML",0.739637


QUERY: I cannot log into my account


,customer_text,response_text,similarity_score
40843,@AmazonHelp I cannot login into my account .com,@375313 Hi Adriana! You've come to the right place. How can we be of service today? ^SY,0.896963
17158,@AmazonHelp I am unable to login,"@182608 Hello, so sorry to hear this. Could you kindly elaborate the issue, so that we can assist you with this.",0.865127
11256,@AmazonHelp I am not able to login,@182868 Sorry for the inconvenience. Contact our support team here: https://t.co/vlvfJr4nN9 we will assist you further. ^GU,0.853878


In [133]:
from src.conversations import build_customer_response_pairs

conversation_pairs = build_customer_response_pairs(
    amazon_customers,
    amazon_tweets
)

print("Conversation pairs:", len(conversation_pairs))

NameError: name 'amazon_tweets' is not defined

In [134]:
import pandas as pd

conversation_pairs = pd.read_csv(
    "../data/processed/amazonhelp_retrieval_pairs.csv"
)

print("Conversation pairs:", len(conversation_pairs))
print(conversation_pairs.columns.tolist())

Conversation pairs: 90708
['customer_tweet_id', 'customer_text', 'response_tweet_id', 'response_text', 'customer_text_clean']


In [135]:
retrieval_eval = golden.merge(
    conversation_pairs[
        ["customer_tweet_id", "response_tweet_id", "response_text"]
    ],
    left_on="tweet_id",
    right_on="customer_tweet_id",
    how="inner"
)

print("Retrieval evaluation examples:", len(retrieval_eval))

Retrieval evaluation examples: 180


In [136]:
import numpy as np

def retrieval_recall_at_k(k=5):
    hits = 0

    for _, row in retrieval_eval.iterrows():

        query_embedding = embedding_model.encode(
            [row["text"]],
            normalize_embeddings=True
        )

        query_embedding = np.asarray(
            query_embedding,
            dtype="float32"
        )

        scores, indices = index.search(
            query_embedding,
            k
        )

        retrieved_ids = retrieval_pairs.iloc[
            indices[0]
        ]["response_tweet_id"].tolist()

        if row["response_tweet_id"] in retrieved_ids:
            hits += 1

    return hits / len(retrieval_eval)

In [137]:
recall_1 = retrieval_recall_at_k(1)
recall_3 = retrieval_recall_at_k(3)
recall_5 = retrieval_recall_at_k(5)

print(f"Recall@1: {recall_1:.2%}")
print(f"Recall@3: {recall_3:.2%}")
print(f"Recall@5: {recall_5:.2%}")

Recall@1: 78.89%
Recall@3: 81.67%
Recall@5: 82.22%


In [138]:
from dotenv import load_dotenv
from google import genai
import os

from src.pipeline import run_support_agent

load_dotenv()

gemini_client = genai.Client(
    api_key=os.getenv("GEMINI_API_KEY")
)

result = run_support_agent(
    customer_message="My package has not arrived yet",
    intent_classifier=semantic_classifier_135,
    embedding_model=embedding_model,
    index=index,
    retrieval_data=retrieval_pairs,
    gemini_client=gemini_client,
    top_k=3
)

print("Intent:", result["intent"])
print("Similarity:", result["similarity_score"])
print("Decision:", result["decision"])
print("Reason:", result["reason"])
print("\nGenerated reply:")
print(result["generated_reply"])

Intent: return_or_cancellation
Similarity: 0.8367633819580078
Decision: auto
Reason: Intent is low-risk and relevant historical evidence was found.

Generated reply:


KeyError: 'generated_reply'

In [139]:
print(result)
print(type(result))

{'customer_message': 'My package has not arrived yet', 'intent': 'return_or_cancellation', 'retrieved_cases':        customer_tweet_id  \
18937             434522   
66087            2135663   
5936              162186   

                                                 customer_text  \
18937            @AmazonHelp My package still hasn’t arrived 😥   
66087             @AmazonHelp Didn't received my package today   
5936   @AmazonHelp My package still hasn’t arrived.  Now what?   

       response_tweet_id  \
18937           434521.0   
66087          2135662.0   
5936            162185.0   

                                                                                                                      response_text  \
18937        @218242 Delivery delays are possible but should be rare. Please let us know if your order doesn't arrive tomorrow! ^ST   
66087                                    @148218 parcel by 21:00 today, please let us know so we can investigate further. ^HC (2/

In [141]:
print(
    semantic_classifier_135.predict_one(
        "My package has not arrived yet",
        top_k=5
    )
)

return_or_cancellation


In [142]:
query = "My package has not arrived yet"

query_embedding = embedding_model.encode(
    [query],
    normalize_embeddings=True
)[0]

scores = np.dot(
    semantic_classifier_135.training_embeddings,
    query_embedding
)

top_indices = np.argsort(scores)[-10:][::-1]

for i in top_indices:
    print(
        f"{scores[i]:.3f} | "
        f"{semantic_classifier_135.training_labels[i]} | "
        f"{semantic_classifier_135.training_labels[i]}"
    )

0.570 | return_or_cancellation | return_or_cancellation
0.544 | order_status | order_status
0.516 | return_or_cancellation | return_or_cancellation
0.515 | delivery_issue | delivery_issue
0.512 | delivery_issue | delivery_issue
0.510 | refund | refund
0.499 | order_status | order_status
0.486 | delivery_issue | delivery_issue
0.485 | delivery_issue | delivery_issue
0.466 | delivery_issue | delivery_issue


In [143]:
for i in top_indices:
    print(
        f"{scores[i]:.3f} | "
        f"{semantic_classifier_135.training_labels[i]} | "
        f"{training_135.iloc[i]['text']}"
    )

0.570 | return_or_cancellation | @AmazonHelp I have placed 3 orders none of them have delivered yet. I had to cancel it after waiting for a months
See Order#__credit_card__
0.544 | order_status | @AmazonHelp Just “Not yet dispatched. We'll e-mail you when we have a delivery date”
0.516 | return_or_cancellation | @AmazonHelp Not helpful! I need the wrong shipment to be picked ASAP, not dropping it anywhere. FIX IT!
0.515 | delivery_issue | @AmazonHelp Nope
It says in the Amazon website,the delivery guy came twice to deliver the pakage but nobody signed to collect the order
Connect by email
0.512 | delivery_issue | @AmazonHelp I already sent an email thourgh your app. Please, I need this package!
0.510 | refund | @AmazonHelp The order says it was delivered yesterday. A refund was issued.
0.499 | order_status | @AmazonHelp It’s been dispatched but it still doesn’t explain why my order was prioritised last
0.486 | delivery_issue | @AmazonHelp Still waiting. Not what I call next day deliver

In [144]:
semantic_classifier_weighted = SemanticIntentClassifier(
    embedding_model
)

semantic_classifier_weighted.train(
    training_135["text"],
    training_135["intent"]
)

print("Weighted semantic classifier trained.")

Weighted semantic classifier trained.


In [145]:
test_message = "My package has not arrived yet"

prediction = semantic_classifier_weighted.predict_one(
    test_message,
    top_k=10
)

print("Prediction:", prediction)

Prediction: delivery_issue


In [146]:
from sklearn.metrics import accuracy_score, classification_report

weighted_predictions = semantic_classifier_weighted.predict(
    golden["text"],
    top_k=10
)

weighted_accuracy = accuracy_score(
    golden["final_intent"],
    weighted_predictions
)

print(f"Weighted Semantic KNN accuracy: {weighted_accuracy:.2%}")

print(
    classification_report(
        golden["final_intent"],
        weighted_predictions,
        zero_division=0
    )
)

Weighted Semantic KNN accuracy: 46.50%
                         precision    recall  f1-score   support

complaint_or_escalation       0.37      0.32      0.34        34
         delivery_issue       0.83      0.57      0.67        51
           order_status       0.67      0.10      0.17        20
                  other       0.40      0.96      0.57        49
     payment_or_billing       0.00      0.00      0.00         8
       prime_membership       0.00      0.00      0.00         6
   product_availability       0.00      0.00      0.00         7
          product_issue       0.00      0.00      0.00         5
                 refund       0.38      0.38      0.38         8
 return_or_cancellation       0.00      0.00      0.00         5
      technical_support       0.50      0.14      0.22         7

               accuracy                           0.47       200
              macro avg       0.29      0.22      0.21       200
           weighted avg       0.47      0.47     

In [147]:
from dotenv import load_dotenv
from google import genai
import os

from src.pipeline import run_support_agent

load_dotenv()

gemini_client = genai.Client(
    api_key=os.getenv("GEMINI_API_KEY")
)

result = run_support_agent(
    customer_message="My package has not arrived yet",
    intent_classifier=semantic_classifier_135,
    embedding_model=embedding_model,
    index=index,
    retrieval_data=retrieval_pairs,
    gemini_client=gemini_client,
    top_k=3
)

print("Customer:", result["customer_message"])
print("Intent:", result["intent"])
print("Similarity:", result["similarity_score"])
print("Decision:", result["decision"])
print("Reason:", result["reason"])
print("\nGenerated reply:")
print(result["generated_reply"])

Customer: My package has not arrived yet
Intent: return_or_cancellation
Similarity: 0.8367633819580078
Decision: auto
Reason: Intent is low-risk and relevant historical evidence was found.

Generated reply:


KeyError: 'generated_reply'

In [148]:
import importlib
import src.pipeline

importlib.reload(src.pipeline)

from src.pipeline import run_support_agent

In [149]:
import inspect

print(inspect.getsource(run_support_agent))

def run_support_agent(
    customer_message,
    intent_classifier,
    embedding_model,
    index,
    retrieval_data,
    gemini_client,
    top_k=3
):
    """
    Run the complete Amazon customer-support pipeline.

    Steps:
    1. Predict intent.
    2. Retrieve similar historical cases.
    3. Generate a grounded reply.
    4. Decide auto-handle vs human escalation.
    """

    # 1. Intent classification
    intent = intent_classifier.predict_one(
        customer_message,
        top_k=5
    )

    # 2. Historical retrieval
    retrieved_cases = retrieve_similar_cases(
        customer_message,
        embedding_model,
        index,
        retrieval_data,
        top_k=top_k
    )

    similarity_score = float(
        retrieved_cases["similarity_score"].iloc[0]
    )

    # 3. Generate grounded reply
    generated_reply = generate_grounded_reply(
        gemini_client,
        customer_message,
        intent,
        retrieved_cases
    )

    # 4. Escalation decision
    e

In [150]:
result = run_support_agent(
    customer_message="My package has not arrived yet",
    intent_classifier=semantic_classifier_135,
    embedding_model=embedding_model,
    index=index,
    retrieval_data=retrieval_pairs,
    gemini_client=gemini_client,
    top_k=3
)

print("Customer:", result["customer_message"])
print("Intent:", result["intent"])
print("Similarity:", result["similarity_score"])
print("Decision:", result["decision"])
print("Reason:", result["reason"])
print("\nGenerated reply:")
print(result["generated_reply"])

Customer: My package has not arrived yet
Intent: return_or_cancellation
Similarity: 0.8367633819580078
Decision: auto
Reason: Intent is low-risk and relevant historical evidence was found.

Generated reply:
We're sorry to hear your package hasn't arrived yet. Delivery delays can sometimes happen. Please let us know if your order still hasn't arrived by the end of tomorrow so we can investigate this further for you.


In [151]:
training_135["intent"].value_counts().sort_values()

intent
prime_membership            3
order_status                4
product_availability        7
payment_or_billing          8
return_or_cancellation      9
product_issue               9
refund                     11
technical_support          13
delivery_issue             22
complaint_or_escalation    22
other                      27
Name: count, dtype: int64

In [152]:
training_135.groupby("intent")["text"].apply(list)

intent
complaint_or_escalation                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                   [@AmazonHelp Did while having a chat on FB, was informed will get back, nothing happened, @AmazonHelp Yeah like I said, useless., @AmazonHelp Please look at complaints registered for Order #407-4330002-1760315 @115850, @AmazonHelp Info provided through the link. Please let me know the phone number that will call me. I do not answer unknown n

In [153]:
import numpy as np
from collections import defaultdict


class PrototypeIntentClassifier:
    """
    Classify an intent by comparing a message
    with the semantic prototype of each intent.
    """

    def __init__(self, embedding_model):
        self.embedding_model = embedding_model
        self.prototypes = {}
        self.labels = []

    def train(self, texts, labels):
        embeddings = self.embedding_model.encode(
            list(texts),
            normalize_embeddings=True
        )

        grouped = defaultdict(list)

        for embedding, label in zip(embeddings, labels):
            grouped[label].append(embedding)

        self.prototypes = {}

        for label, vectors in grouped.items():
            centroid = np.mean(vectors, axis=0)

            # Normalize centroid
            centroid = centroid / np.linalg.norm(centroid)

            self.prototypes[label] = centroid

        self.labels = list(self.prototypes.keys())

    def predict_one(self, text):
        query_embedding = self.embedding_model.encode(
            [text],
            normalize_embeddings=True
        )[0]

        scores = {}

        for label, prototype in self.prototypes.items():
            scores[label] = np.dot(
                query_embedding,
                prototype
            )

        return max(
            scores,
            key=scores.get
        )

    def predict(self, texts):
        return [
            self.predict_one(text)
            for text in texts
        ]

In [154]:
prototype_classifier = PrototypeIntentClassifier(
    embedding_model
)

prototype_classifier.train(
    training_135["text"],
    training_135["intent"]
)

print("Prototype classifier trained.")

Prototype classifier trained.


In [155]:
prediction = prototype_classifier.predict_one(
    "My package has not arrived yet"
)

print("Prediction:", prediction)

Prediction: delivery_issue


In [156]:
from sklearn.metrics import accuracy_score, classification_report

prototype_predictions = prototype_classifier.predict(
    golden["text"]
)

prototype_accuracy = accuracy_score(
    golden["final_intent"],
    prototype_predictions
)

print(f"Prototype classifier accuracy: {prototype_accuracy:.2%}")

print(
    classification_report(
        golden["final_intent"],
        prototype_predictions,
        zero_division=0
    )
)

Prototype classifier accuracy: 54.50%
                         precision    recall  f1-score   support

complaint_or_escalation       0.46      0.62      0.53        34
         delivery_issue       0.84      0.61      0.70        51
           order_status       0.20      0.05      0.08        20
                  other       0.63      0.80      0.70        49
     payment_or_billing       0.00      0.00      0.00         8
       prime_membership       0.25      0.17      0.20         6
   product_availability       0.50      0.57      0.53         7
          product_issue       0.20      0.20      0.20         5
                 refund       0.45      0.62      0.53         8
 return_or_cancellation       0.50      0.60      0.55         5
      technical_support       0.43      0.43      0.43         7

               accuracy                           0.55       200
              macro avg       0.41      0.42      0.40       200
           weighted avg       0.54      0.55      

In [157]:
from sklearn.metrics import confusion_matrix
import pandas as pd

labels = sorted(golden["final_intent"].unique())

cm = confusion_matrix(
    golden["final_intent"],
    prototype_predictions,
    labels=labels
)

confusion_df = pd.DataFrame(
    cm,
    index=labels,
    columns=labels
)

confusion_df

,complaint_or_escalation,delivery_issue,order_status,other,payment_or_billing,prime_membership,product_availability,product_issue,refund,return_or_cancellation,technical_support
complaint_or_escalation,21,1,1,6,1,0,0,2,0,0,2
delivery_issue,2,31,3,7,2,1,1,1,1,2,0
order_status,4,3,1,7,3,2,0,0,0,0,0
other,5,1,0,39,1,0,1,0,2,0,0
payment_or_billing,4,0,0,0,0,0,1,1,2,0,0
prime_membership,2,0,0,0,1,1,0,0,1,0,1
product_availability,1,0,0,2,0,0,4,0,0,0,0
product_issue,2,0,0,0,1,0,0,1,0,1,0
refund,2,0,0,0,0,0,0,0,5,0,1
return_or_cancellation,1,1,0,0,0,0,0,0,0,3,0


In [158]:
errors = golden[
    golden["final_intent"] != prototype_predictions
].copy()

errors["predicted_intent"] = [
    prototype_predictions[i]
    for i in errors.index
]

errors[
    ["text", "final_intent", "predicted_intent"]
].head(20)

,text,final_intent,predicted_intent
12,@AmazonHelp No resolution provided. Still facing issue,complaint_or_escalation,technical_support
15,@AmazonHelp Yes reordered no delivery date and customer services advised they'd speak to the relevant department however 9 days later still no shipment.,delivery_issue,order_status
17,@AmazonHelp What is the status still no response,order_status,complaint_or_escalation
18,@AmazonHelp I want it today else cancel it,delivery_issue,return_or_cancellation
19,@AmazonHelp I filled the form yesterday and still waiting for YOU to revert. 3rd day of #amazonprime delivery.#Joke,delivery_issue,complaint_or_escalation
22,@AmazonHelp It was fulfilled by Amazon with Prime Shipping.,order_status,prime_membership
23,@AmazonHelp Not yet but will not be surprised due to the fact that it has not yet been dispatched,delivery_issue,order_status
25,@AmazonHelp I’ve had multiple bad experiences this year and I’ve rolled with it but it’s getting utterly ridiculous. Phone support was no help at all,complaint_or_escalation,product_issue
27,@AmazonHelp I am No longer a prime member,prime_membership,complaint_or_escalation
29,"@AmazonHelp 2nd day in a row. Do you not use safe places anymore. Why are people paying for a prime service, when we don't get one. https://t.co/IgOLLLbE6i",delivery_issue,prime_membership


In [159]:
import importlib
import src.semantic_intents

importlib.reload(src.semantic_intents)

from src.semantic_intents import PrototypeIntentClassifier

In [160]:
prototype_classifier = PrototypeIntentClassifier(
    embedding_model
)

prototype_classifier.train(
    training_135["text"],
    training_135["intent"]
)

print("Prototype classifier trained successfully.")

Prototype classifier trained successfully.


In [161]:
print(
    prototype_classifier.predict_one(
        "My package has not arrived yet"
    )
)

delivery_issue


In [162]:
from src.pipeline import run_support_agent

In [163]:
import importlib
import src.pipeline

importlib.reload(src.pipeline)

from src.pipeline import run_support_agent

In [164]:
result = run_support_agent(
    customer_message="My package has not arrived yet",
    intent_classifier=prototype_classifier,
    embedding_model=embedding_model,
    index=index,
    retrieval_data=retrieval_pairs,
    gemini_client=gemini_client,
    top_k=3
)

print("Customer:", result["customer_message"])
print("Intent:", result["intent"])
print("Similarity:", result["similarity_score"])
print("Decision:", result["decision"])
print("Reason:", result["reason"])
print("\nGenerated reply:")
print(result["generated_reply"])

TypeError: PrototypeIntentClassifier.predict_one() got an unexpected keyword argument 'top_k'

In [165]:
import importlib
import src.pipeline

importlib.reload(src.pipeline)

from src.pipeline import run_support_agent

In [168]:
result = run_support_agent(
    customer_message="My package has not arrived yet",
    intent_classifier=prototype_classifier,
    embedding_model=embedding_model,
    index=index,
    retrieval_data=retrieval_pairs,
    gemini_client=gemini_client,
    top_k=3
)

print("Customer:", result["customer_message"])
print("Intent:", result["intent"])
print("Similarity:", result["similarity_score"])
print("Decision:", result["decision"])
print("Reason:", result["reason"])
print("\nGenerated reply:")
print(result["generated_reply"])

Customer: My package has not arrived yet
Intent: delivery_issue
Similarity: 0.8367633819580078
Decision: auto
Reason: Intent is low-risk and relevant historical evidence was found.

Generated reply:
Thanks for letting us know about your package. Delivery delays can sometimes occur. Please let us know if your order doesn't arrive by tomorrow, and we'll be happy to investigate further for you.


In [170]:
import importlib
import src.generation

importlib.reload(src.generation)

<module 'src.generation' from 'd:\\Hiver_Assigenment\\src\\generation.py'>

In [172]:
import importlib
import src.generation

importlib.reload(src.generation)

<module 'src.generation' from 'd:\\Hiver_Assigenment\\src\\generation.py'>

In [173]:
result = run_support_agent(
    customer_message="My package has not arrived yet",
    intent_classifier=prototype_classifier,
    embedding_model=embedding_model,
    index=index,
    retrieval_data=retrieval_pairs,
    gemini_client=gemini_client,
    top_k=3
)

print("Customer:", result["customer_message"])
print("Intent:", result["intent"])
print("Similarity:", result["similarity_score"])
print("Decision:", result["decision"])
print("Reason:", result["reason"])
print("\nGenerated reply:")
print(result["generated_reply"])

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 37.328776675s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'model': 'gemini-2.5-flash', 'location': 'global'},

In [174]:
from src.generation import sanitize_generated_reply

test_reply = (
    "Thanks for letting us know about your package. "
    "Delivery delays can sometimes occur. "
    "Please let us know if your order doesn't arrive by tomorrow, "
    "and we'll be happy to investigate further for you."
)

cleaned = sanitize_generated_reply(
    test_reply,
    "My package has not arrived yet"
)

print(cleaned)from src.generation import sanitize_generated_reply

test_reply = (
    "Thanks for letting us know about your package. "
    "Delivery delays can sometimes occur. "
    "Please let us know if your order doesn't arrive by tomorrow, "
    "and we'll be happy to investigate further for you."
)

cleaned = sanitize_generated_reply(
    test_reply,
    "My package has not arrived yet"
)

print(cleaned)

SyntaxError: invalid syntax (2768986270.py, line 15)

In [175]:
from src.generation import sanitize_generated_reply

test_reply = (
    "Thanks for letting us know about your package. "
    "Delivery delays can sometimes occur. "
    "Please let us know if your order doesn't arrive by tomorrow, "
    "and we'll be happy to investigate further for you."
)

cleaned = sanitize_generated_reply(
    test_reply,
    "My package has not arrived yet"
)

print(cleaned)

Thanks for letting us know about your package. Delivery delays can sometimes occur.


In [176]:
import importlib
import src.pipeline
import src.generation

importlib.reload(src.generation)
importlib.reload(src.pipeline)

print(src.generation.generate_grounded_reply.__module__)
print(src.pipeline.generate_grounded_reply.__module__)

src.generation
src.generation


In [177]:
from evaluation.classification_metrics import (
    evaluate_classification,
    get_classification_report,
)

y_true = [
    "delivery_issue",
    "refund",
    "delivery_issue",
    "other",
]

y_pred = [
    "delivery_issue",
    "refund",
    "other",
    "other",
]

metrics = evaluate_classification(
    y_true,
    y_pred
)

print(metrics)

print("\nClassification Report:")
print(
    get_classification_report(
        y_true,
        y_pred
    )
)

{'accuracy': 0.75, 'macro_f1': 0.7777777777777777, 'weighted_f1': 0.75}

Classification Report:
                precision    recall  f1-score   support

delivery_issue       1.00      0.50      0.67         2
         other       0.50      1.00      0.67         1
        refund       1.00      1.00      1.00         1

      accuracy                           0.75         4
     macro avg       0.83      0.83      0.78         4
  weighted avg       0.88      0.75      0.75         4



In [178]:
from evaluation.evaluate import evaluate_classifier

metrics, report, predictions = evaluate_classifier(
    classifier=prototype_classifier,
    golden_path="../data/golden/amazonhelp_golden.csv"
)

print("Metrics:")
print(metrics)

print("\nClassification Report:")
print(report)

print("\nPredictions:")
print(predictions[
    [
        "tweet_id",
        "text",
        "final_intent",
        "predicted_intent",
        "correct"
    ]
].head())

Metrics:
{'accuracy': 0.545, 'macro_f1': 0.4041748412801045, 'weighted_f1': 0.5284269149532309}

Classification Report:
                         precision    recall  f1-score   support

complaint_or_escalation       0.46      0.62      0.53        34
         delivery_issue       0.84      0.61      0.70        51
           order_status       0.20      0.05      0.08        20
                  other       0.63      0.80      0.70        49
     payment_or_billing       0.00      0.00      0.00         8
       prime_membership       0.25      0.17      0.20         6
   product_availability       0.50      0.57      0.53         7
          product_issue       0.20      0.20      0.20         5
                 refund       0.45      0.62      0.53         8
 return_or_cancellation       0.50      0.60      0.55         5
      technical_support       0.43      0.43      0.43         7

               accuracy                           0.55       200
              macro avg       0.4

In [179]:
from evaluation.retrieval_metrics import evaluate_retrieval

retrieval_results = evaluate_retrieval(
    evaluation_data=retrieval_eval,
    retrieval_data=retrieval_pairs,
    embedding_model=embedding_model,
    index=index,
    ks=(1, 3, 5)
)

print(retrieval_results)

{'recall@1': 0.7888888888888889, 'recall@3': 0.8166666666666667, 'recall@5': 0.8222222222222222}


In [180]:
from evaluation.failure_analysis import (
    find_confusion_pairs,
    get_top_failure_modes,
)

print("Top confusion pairs:")

confusions = find_confusion_pairs(
    predictions
)

for pair, count in confusions[:10]:
    print(
        f"{pair[0]} -> {pair[1]}: {count}"
    )

Top confusion pairs:
order_status -> other: 7
delivery_issue -> other: 7
complaint_or_escalation -> other: 6
other -> complaint_or_escalation: 5
order_status -> complaint_or_escalation: 4
payment_or_billing -> complaint_or_escalation: 4
delivery_issue -> order_status: 3
order_status -> delivery_issue: 3
order_status -> payment_or_billing: 3
complaint_or_escalation -> technical_support: 2


In [181]:
failure_modes = get_top_failure_modes(
    predictions,
    top_n=5,
    examples_per_mode=3
)

for mode in failure_modes:

    print("\n" + "=" * 60)

    print(
        f"True intent: {mode['true_intent']}"
    )

    print(
        f"Predicted intent: {mode['predicted_intent']}"
    )

    print(
        f"Errors: {mode['error_count']}"
    )

    print("\nExamples:")

    for _, row in mode["examples"].iterrows():

        print(
            f"- [{row['tweet_id']}] "
            f"{row['text']}"
        )


True intent: order_status
Predicted intent: other
Errors: 7

Examples:
- [493427] @AmazonHelp Monday I think
- [1464284] @AmazonHelp 8-9 nov
- [2954521] @AmazonHelp 29/11/2017Votre colis est en cours de livraison

True intent: delivery_issue
Predicted intent: other
Errors: 7

Examples:
- [930108] @AmazonHelp Ich soll mich morgen nochmal melden, wenn die Sendung weiterhin nicht ausgeliefert werden sollte.
- [140066] @AmazonHelp Voici : C20025897313 Parceque j'ai beau me plaindre à chaque fois (genre colis ouvert pour rentrer les articles 1 par 1 dans ma BaL), vous m'expediez de + en + par Amz Logistics et ça ne va pas mieux. Je vais bien finir par vous quitter hein…
- [133756] @AmazonHelp Ja, die wissen auch nicht mehr weiter. Der Mitarbeiter gerade hat gesagt, dass er nicht weiß, warum es nicht verschickt wird. 😂

True intent: complaint_or_escalation
Predicted intent: other
Errors: 6

Examples:
- [1388465] @AmazonHelp Ça fait un moment que je remonte des dysfonctionnements... Pourtant

In [182]:
confusions = find_confusion_pairs(predictions)

print("TOP 5 FAILURE MODES")
print("=" * 60)

for i, (pair, count) in enumerate(confusions[:5], start=1):
    print(
        f"{i}. {pair[0]} -> {pair[1]}: {count} errors"
    )

TOP 5 FAILURE MODES
1. order_status -> other: 7 errors
2. delivery_issue -> other: 7 errors
3. complaint_or_escalation -> other: 6 errors
4. other -> complaint_or_escalation: 5 errors
5. order_status -> complaint_or_escalation: 4 errors


In [183]:
import pandas as pd

baseline_results = pd.DataFrame({
    "model": [
        "Majority baseline",
        "TF-IDF + Logistic Regression",
        "Semantic Prototype Classifier"
    ],
    "accuracy": [
        0.245,
        0.380,
        0.545
    ],
    "macro_f1": [
        None,
        0.11,
        0.404
    ],
    "weighted_f1": [
        None,
        None,
        0.528
    ]
})

baseline_results.to_csv(
    "../results/classification_baselines.csv",
    index=False
)

print(baseline_results)

                           model  accuracy  macro_f1  weighted_f1
0              Majority baseline     0.245       NaN          NaN
1   TF-IDF + Logistic Regression     0.380     0.110          NaN
2  Semantic Prototype Classifier     0.545     0.404        0.528


In [1]:
from google import genai

test_client = genai.Client()

response = test_client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Reply with exactly: QUOTA_OK"
)

print(response.text)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


QUOTA_OK


In [2]:
result = run_support_agent(
    customer_message="My package has not arrived yet",
    intent_classifier=prototype_classifier,
    embedding_model=embedding_model,
    index=index,
    retrieval_data=retrieval_pairs,
    gemini_client=gemini_client,
    top_k=3
)

print("Customer:", result["customer_message"])
print("Intent:", result["intent"])
print("Similarity:", result["similarity_score"])
print("Decision:", result["decision"])
print("Reason:", result["reason"])
print("Generated reply:", result["generated_reply"])

NameError: name 'run_support_agent' is not defined

In [3]:
from src.pipeline import run_support_agent

print("run_support_agent loaded successfully")

ModuleNotFoundError: No module named 'src'

In [4]:
import sys
import os

project_root = os.path.abspath("..")

if project_root not in sys.path:
    sys.path.append(project_root)

print(project_root)
print("Project root added successfully")

d:\Hiver_Assigenment
Project root added successfully


In [5]:
from src.pipeline import run_support_agent

print("run_support_agent loaded successfully")

run_support_agent loaded successfully


In [6]:
from src.pipeline import run_support_agent

print("run_support_agent loaded successfully")

run_support_agent loaded successfully


In [7]:
import pandas as pd

retrieval_pairs = pd.read_csv(
    "../data/processed/amazonhelp_retrieval_pairs.csv"
)

print("Retrieval pairs:", len(retrieval_pairs))
print(retrieval_pairs.head(2))

Retrieval pairs: 90708
   customer_tweet_id                                      customer_text  \
0                270      @AmazonHelp ありがとうございます。\n今、電話で主人が対応していただいてます。   
1                271  @AmazonHelp 電話で対応してもらいましたが改良されませんでした。\n保証期間も過ぎ...   

   response_tweet_id                                      response_text  \
0              269.0  @115770 こんにちは、アマゾン公式です。Fire TV Stickが見れないというのは...   
1              269.0  @115770 こんにちは、アマゾン公式です。Fire TV Stickが見れないというのは...   

                                 customer_text_clean  
0                   ありがとうございます。 今、電話で主人が対応していただいてます。  
1  電話で対応してもらいましたが改良されませんでした。 保証期間も過ぎてるので買い直しになるんで...  


In [8]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully")

d:\Hiver_Assigenment\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2234.44it/s]


Embedding model loaded successfully


In [9]:
import numpy as np
import faiss

embedding_matrix = np.asarray(
    embedding_model.encode(
        retrieval_pairs["customer_text_clean"].tolist(),
        normalize_embeddings=True,
        show_progress_bar=True
    ),
    dtype="float32"
)

index = faiss.IndexFlatIP(
    embedding_matrix.shape[1]
)

index.add(embedding_matrix)

print("FAISS index loaded successfully")
print("Number of vectors:", index.ntotal)

Batches: 100%|██████████| 2835/2835 [13:02<00:00,  3.62it/s]


FAISS index loaded successfully
Number of vectors: 90708


In [10]:
import pandas as pd

training_135 = pd.read_csv(
    "../data/processed/amazonhelp_training_135.csv"
)

print("Training examples:", len(training_135))
print(training_135["final_intent"].value_counts())

Training examples: 135


KeyError: 'final_intent'

In [11]:
print(training_135.columns.tolist())
print(training_135.head())

['tweet_id', 'author_id', 'inbound', 'created_at', 'text', 'response_tweet_id', 'in_response_to_tweet_id', 'intent']
   tweet_id  author_id  inbound                      created_at  \
0    315275     191151     True  Sat Oct 07 08:00:34 +0000 2017   
1    668492     279394     True  Thu Nov 23 04:25:22 +0000 2017   
2    312989     190746     True  Sat Oct 07 07:41:25 +0000 2017   
3     27898     122104     True  Wed Nov 01 11:29:39 +0000 2017   
4    304638     124163     True  Sat Nov 04 04:13:51 +0000 2017   

                                                text     response_tweet_id  \
0  @AmazonHelp @115850 thanks my query has been r...                315277   
1  @AmazonHelp Then why you claim of next day del...                668494   
2  @AmazonHelp Did while having a chat on FB, was...  312990,312991,312992   
3  @AmazonHelp Are you going to be offering anyth...                   NaN   
4  @AmazonHelp Not helpful! I need the wrong ship...                304636   

   in_respo

In [12]:
from src.semantic_intents import PrototypeIntentClassifier

prototype_classifier = PrototypeIntentClassifier(
    embedding_model=embedding_model
)

prototype_classifier.train(
    training_135["text"].fillna("").tolist(),
    training_135["intent"].tolist()
)

print("Prototype classifier trained successfully")
print("Number of intents:", len(prototype_classifier.labels))
print("Intents:", prototype_classifier.labels)

Prototype classifier trained successfully
Number of intents: 11
Intents: ['other', 'delivery_issue', 'complaint_or_escalation', 'return_or_cancellation', 'prime_membership', 'product_issue', 'technical_support', 'order_status', 'refund', 'payment_or_billing', 'product_availability']


In [13]:
import os
from google import genai
from dotenv import load_dotenv

load_dotenv()

gemini_client = genai.Client(
    api_key=os.getenv("GEMINI_API_KEY")
)

print("Gemini client loaded successfully")

Gemini client loaded successfully


In [14]:
result = run_support_agent(
    customer_message="My package has not arrived yet",
    intent_classifier=prototype_classifier,
    embedding_model=embedding_model,
    index=index,
    retrieval_data=retrieval_pairs,
    gemini_client=gemini_client,
    top_k=3
)

print("Customer:", result["customer_message"])
print("Intent:", result["intent"])
print("Similarity:", result["similarity_score"])
print("Decision:", result["decision"])
print("Reason:", result["reason"])
print("Generated reply:", result["generated_reply"])

Customer: My package has not arrived yet
Intent: delivery_issue
Similarity: 0.8367633819580078
Decision: auto
Reason: Intent is low-risk and relevant historical evidence was found.
Generated reply: Thanks for reaching out! We understand your package has not arrived. Please let us know if it still hasn't arrived, and we'd be happy to help.


In [15]:
from evaluation.reply_judge import judge_reply

judge_result = judge_reply(
    client=gemini_client,
    customer_message=result["customer_message"],
    generated_reply=result["generated_reply"],
    retrieved_cases=result["retrieved_cases"]
)

print(judge_result)

{'helpfulness': 1, 'groundedness': 1, 'unsupported_claims': 5, 'professional_tone': 4, 'overall_score': 2, 'reason': "The reply is very unhelpful as it offers no concrete next steps or estimated resolution. It merely acknowledges the problem and then asks the customer to re-initiate contact if the problem persists, without providing any timeframe. This is directly contrary to the historical evidence, which consistently provides a specific waiting period ('tomorrow', 'by 21:00 today', 'by Monday') before further action. Therefore, it is poorly grounded as it misses the critical instruction from historical examples. While the tone is professional and no false claims are made, the complete lack of actionable guidance makes it ineffective."}


In [16]:
print(
    result["retrieved_cases"][
        [
            "customer_text",
            "response_text",
            "similarity_score"
        ]
    ].to_string(index=False)
)

                                          customer_text                                                                                                                response_text  similarity_score
          @AmazonHelp My package still hasn’t arrived 😥       @218242 Delivery delays are possible but should be rare. Please let us know if your order doesn't arrive tomorrow! ^ST          0.836763
           @AmazonHelp Didn't received my package today                                   @148218 parcel by 21:00 today, please let us know so we can investigate further. ^HC (2/2)          0.809228
@AmazonHelp My package still hasn’t arrived.  Now what? @153437 Thank you for reaching out to us! Please let us know if your package does not arrive by Monday. We want to help! ^SM          0.797952


In [17]:
import importlib
import evaluation.reply_judge

importlib.reload(evaluation.reply_judge)

<module 'evaluation.reply_judge' from 'd:\\Hiver_Assigenment\\evaluation\\reply_judge.py'>

In [18]:
judge_result = evaluation.reply_judge.judge_reply(
    client=gemini_client,
    customer_message=result["customer_message"],
    generated_reply=result["generated_reply"],
    retrieved_cases=result["retrieved_cases"]
)

print(judge_result)

{'helpfulness': 2, 'groundedness': 5, 'unsupported_claims': 5, 'professional_tone': 5, 'overall_score': 4, 'reason': "The AI's reply maintains a professional and empathetic tone ('Thanks for reaching out! We understand...') and makes no unsupported claims. It is also perfectly grounded according to the provided evaluation rules: it correctly extracts the general resolution pattern of asking the customer to re-contact if the package still hasn't arrived, without transferring any case-specific, non-transferable details such as specific dates or times (as explicitly demonstrated in the 'General resolution pattern' example). However, by omitting any timeframe, the reply's helpfulness is significantly reduced. While it avoids transferring specific dates, it also fails to provide the customer with any useful guidance on *when* to expect the package or *when* to follow up again, leaving them without clear next steps or expectations."}


In [19]:
from evaluation.human_agreement import (
    calculate_human_agreement
)

human_labels = [
    "acceptable",
    "acceptable",
    "not_acceptable",
    "acceptable",
    "not_acceptable"
]

judge_scores = [
    4,
    5,
    2,
    4,
    1
]

agreement_result = calculate_human_agreement(
    human_labels,
    judge_scores
)

print(agreement_result)

{'judge_labels': ['acceptable', 'acceptable', 'not_acceptable', 'acceptable', 'not_acceptable'], 'agreement': 1.0, 'cohen_kappa': 1.0}


In [20]:
import pandas as pd

golden = pd.read_csv(
    "../data/golden/amazonhelp_golden.csv"
)

# Select up to 2 examples from each intent
human_eval_sample = (
    golden
    .groupby("final_intent", group_keys=False)
    .head(2)
    .reset_index(drop=True)
)

print("Number of examples:", len(human_eval_sample))

print("\nIntent distribution:")
print(
    human_eval_sample["final_intent"]
    .value_counts()
)

print("\nExamples:")
print(
    human_eval_sample[
        ["tweet_id", "text", "final_intent"]
    ].to_string(index=False)
)

Number of examples: 22

Intent distribution:
final_intent
delivery_issue             2
other                      2
complaint_or_escalation    2
order_status               2
product_availability       2
prime_membership           2
refund                     2
return_or_cancellation     2
payment_or_billing         2
product_issue              2
technical_support          2
Name: count, dtype: int64

Examples:
 tweet_id                                                                                                                                                                         text            final_intent
  2470126       @AmazonHelp Something was supposed to be delivered today but it says on the order page ‘arrival at incorrect carrier facility’ and to ‘check back tuesday’ (yesterday)          delivery_issue
  1705395                                                                                                                                                     @AmazonHelp Th

In [21]:
human_eval_sample.to_csv(
    "../data/golden/reply_quality_sample.csv",
    index=False
)

print("Saved:", len(human_eval_sample), "examples")

Saved: 22 examples


In [22]:
import pandas as pd

sample = pd.read_csv(
    "../data/golden/reply_quality_sample.csv"
)

pilot_sample = sample.head(4).copy()

print(pilot_sample[
    ["tweet_id", "text", "final_intent"]
].to_string(index=False))

 tweet_id                                                                                                                                                                   text   final_intent
  2470126 @AmazonHelp Something was supposed to be delivered today but it says on the order page ‘arrival at incorrect carrier facility’ and to ‘check back tuesday’ (yesterday) delivery_issue
  1705395                                                                                                                                               @AmazonHelp Thank you 🙌🏻          other
  2957011                                                                                                                                     @AmazonHelp C’est Amazon Logistics          other
    55058                @AmazonHelp yes I have been through all of the steps. I will check with nighbours further down the street tomorrow, but no notification of delivery 1/2 delivery_issue


In [23]:
pilot_results = []

for _, row in pilot_sample.iterrows():

    result = run_support_agent(
        customer_message=row["text"],
        intent_classifier=prototype_classifier,
        embedding_model=embedding_model,
        index=index,
        retrieval_data=retrieval_pairs,
        gemini_client=gemini_client,
        top_k=3
    )

    pilot_results.append({
        "tweet_id": row["tweet_id"],
        "customer_message": row["text"],
        "gold_intent": row["final_intent"],
        "predicted_intent": result["intent"],
        "similarity_score": result["similarity_score"],
        "generated_reply": result["generated_reply"],
        "decision": result["decision"],
        "decision_reason": result["reason"]
    })

pilot_results_df = pd.DataFrame(pilot_results)

pilot_results_df.to_csv(
    "../results/reply_quality_pilot.csv",
    index=False
)

print("Generated replies:", len(pilot_results_df))

display(
    pilot_results_df[
        [
            "tweet_id",
            "gold_intent",
            "predicted_intent",
            "similarity_score",
            "generated_reply",
            "decision"
        ]
    ]
)

Generated replies: 4


,tweet_id,gold_intent,predicted_intent,similarity_score,generated_reply,decision
0,2470126,delivery_issue,delivery_issue,0.934473,"@ Hi, thanks for getting in touch. We understa...",auto
1,1705395,other,other,0.712925,We're happy to help!,auto
2,2957011,other,other,0.793177,Bonjour. Pourriez-vous nous indiquer quel est ...,auto
3,55058,delivery_issue,delivery_issue,0.957437,We're here to help! We understand you've alrea...,auto


In [24]:
for _, row in pilot_results_df.iterrows():
    print("=" * 80)
    print("Tweet ID:", row["tweet_id"])
    print("Gold intent:", row["gold_intent"])
    print("Predicted intent:", row["predicted_intent"])
    print("Similarity:", row["similarity_score"])
    print("Decision:", row["decision"])
    print("Reason:", row["decision_reason"])
    print("\nCustomer:")
    print(row["customer_message"])
    print("\nGenerated reply:")
    print(row["generated_reply"])
    print()

Tweet ID: 2470126
Gold intent: delivery_issue
Predicted intent: delivery_issue
Similarity: 0.9344730973243713
Decision: auto
Reason: Intent is low-risk and relevant historical evidence was found.

Customer:
@AmazonHelp Something was supposed to be delivered today but it says on the order page ‘arrival at incorrect carrier facility’ and to ‘check back tuesday’ (yesterday)

Generated reply:
@ Hi, thanks for getting in touch. We understand you're seeing an 'arrival at incorrect carrier facility' status. To help us look into this for you without posting any of your account details, can you tell us how we can best assist?

Tweet ID: 1705395
Gold intent: other
Predicted intent: other
Similarity: 0.712925374507904
Decision: auto
Reason: Intent is low-risk and relevant historical evidence was found.

Customer:
@AmazonHelp Thank you 🙌🏻

Generated reply:
We're happy to help!

Tweet ID: 2957011
Gold intent: other
Predicted intent: other
Similarity: 0.7931772470474243
Decision: auto
Reason: Intent

In [25]:
for i, row in pilot_results_df.iterrows():

    print("\n" + "=" * 100)
    print(f"CASE {i + 1}")
    print("=" * 100)

    print("Tweet ID:", row["tweet_id"])
    print("Gold intent:", row["gold_intent"])
    print("Predicted intent:", row["predicted_intent"])
    print("Similarity:", round(row["similarity_score"], 3))
    print("Decision:", row["decision"])

    print("\nCUSTOMER MESSAGE:")
    print(row["customer_message"])

    print("\nGENERATED REPLY:")
    print(row["generated_reply"])


CASE 1
Tweet ID: 2470126
Gold intent: delivery_issue
Predicted intent: delivery_issue
Similarity: 0.934
Decision: auto

CUSTOMER MESSAGE:
@AmazonHelp Something was supposed to be delivered today but it says on the order page ‘arrival at incorrect carrier facility’ and to ‘check back tuesday’ (yesterday)

GENERATED REPLY:
@ Hi, thanks for getting in touch. We understand you're seeing an 'arrival at incorrect carrier facility' status. To help us look into this for you without posting any of your account details, can you tell us how we can best assist?

CASE 2
Tweet ID: 1705395
Gold intent: other
Predicted intent: other
Similarity: 0.713
Decision: auto

CUSTOMER MESSAGE:
@AmazonHelp Thank you 🙌🏻

GENERATED REPLY:
We're happy to help!

CASE 3
Tweet ID: 2957011
Gold intent: other
Predicted intent: other
Similarity: 0.793
Decision: auto

CUSTOMER MESSAGE:
@AmazonHelp C’est Amazon Logistics

GENERATED REPLY:
Bonjour. Pourriez-vous nous indiquer quel est le transporteur, en vérifiant l'e-mail

In [26]:
test_message = "@AmazonHelp Thank you 🙌🏻"

test_result = run_support_agent(
    customer_message=test_message,
    intent_classifier=prototype_classifier,
    embedding_model=embedding_model,
    index=index,
    retrieval_data=retrieval_pairs,
    gemini_client=gemini_client,
    top_k=3
)

print("Customer:")
print(test_result["customer_message"])

print("\nIntent:")
print(test_result["intent"])

print("\nSimilarity:")
print(test_result["similarity_score"])

print("\nDecision:")
print(test_result["decision"])

print("\nReason:")
print(test_result["reason"])

print("\nGenerated reply:")
print(test_result["generated_reply"])

Customer:
@AmazonHelp Thank you 🙌🏻

Intent:
other

Similarity:
0.712925374507904

Decision:
auto

Reason:
Intent is low-risk and relevant historical evidence was found.

Generated reply:
You're welcome! We're here to help if you need anything else. ^RC


In [27]:
pilot_results_df.loc[
    pilot_results_df["tweet_id"] == 1705395,
    "generated_reply"
] = test_result["generated_reply"]

pilot_results_df.loc[
    pilot_results_df["tweet_id"] == 1705395,
    "predicted_intent"
] = test_result["intent"]

pilot_results_df.loc[
    pilot_results_df["tweet_id"] == 1705395,
    "similarity_score"
] = test_result["similarity_score"]

pilot_results_df.loc[
    pilot_results_df["tweet_id"] == 1705395,
    "decision"
] = test_result["decision"]

pilot_results_df.loc[
    pilot_results_df["tweet_id"] == 1705395,
    "decision_reason"
] = test_result["reason"]

pilot_results_df.to_csv(
    "../results/reply_quality_pilot.csv",
    index=False
)

print("Case 2 updated successfully.")

Case 2 updated successfully.


In [28]:
for _, row in pilot_results_df.iterrows():
    if row["tweet_id"] != 1705395:
        print("\n" + "=" * 80)
        print("Tweet ID:", row["tweet_id"])
        print("Customer:", row["customer_message"])
        print("Gold intent:", row["gold_intent"])
        print("Predicted intent:", row["predicted_intent"])
        print("Generated reply:", row["generated_reply"])
        print("Decision:", row["decision"])


Tweet ID: 2470126
Customer: @AmazonHelp Something was supposed to be delivered today but it says on the order page ‘arrival at incorrect carrier facility’ and to ‘check back tuesday’ (yesterday)
Gold intent: delivery_issue
Predicted intent: delivery_issue
Generated reply: @ Hi, thanks for getting in touch. We understand you're seeing an 'arrival at incorrect carrier facility' status. To help us look into this for you without posting any of your account details, can you tell us how we can best assist?
Decision: auto

Tweet ID: 2957011
Customer: @AmazonHelp C’est Amazon Logistics
Gold intent: other
Predicted intent: other
Generated reply: Bonjour. Pourriez-vous nous indiquer quel est le transporteur, en vérifiant l'e-mail d'expédition ou l'emballage de votre colis?
Decision: auto

Tweet ID: 55058
Customer: @AmazonHelp yes I have been through all of the steps. I will check with nighbours further down the street tomorrow, but no notification of delivery 1/2
Gold intent: delivery_issue
Pre

In [30]:
from src.retrieval import retrieve_similar_cases
from evaluation.reply_judge import judge_reply
import pandas as pd

print("Imports successful")

Imports successful


In [31]:
from evaluation.reply_judge import judge_reply

judge_results = []

for _, row in pilot_results_df.iterrows():

    retrieved_cases = retrieve_similar_cases(
        row["customer_message"],
        embedding_model,
        index,
        retrieval_pairs,
        top_k=3
    )

    result = judge_reply(
        client=gemini_client,
        customer_message=row["customer_message"],
        generated_reply=row["generated_reply"],
        retrieved_cases=retrieved_cases
    )

    judge_results.append({
        "tweet_id": row["tweet_id"],
        "customer_message": row["customer_message"],
        "generated_reply": row["generated_reply"],
        "helpfulness": result["helpfulness"],
        "groundedness": result["groundedness"],
        "unsupported_claims": result["unsupported_claims"],
        "professional_tone": result["professional_tone"],
        "overall_score": result["overall_score"],
        "reason": result["reason"]
    })

judge_results_df = pd.DataFrame(judge_results)

judge_results_df.to_csv(
    "../results/reply_quality_judge_pilot.csv",
    index=False
)

display(judge_results_df)

,tweet_id,customer_message,generated_reply,helpfulness,groundedness,unsupported_claims,professional_tone,overall_score,reason
0,2470126,@AmazonHelp Something was supposed to be deliv...,"@ Hi, thanks for getting in touch. We understa...",4,5,5,5,5,The AI reply is excellent. It accurately ackno...
1,1705395,@AmazonHelp Thank you 🙌🏻,You're welcome! We're here to help if you need...,5,5,5,5,5,The AI reply perfectly addresses the customer'...
2,2957011,@AmazonHelp C’est Amazon Logistics,Bonjour. Pourriez-vous nous indiquer quel est ...,4,5,5,5,5,"The AI reply, while seemingly asking for infor..."
3,55058,@AmazonHelp yes I have been through all of the...,We're here to help! We understand you've alrea...,4,1,5,5,3,The AI reply demonstrates good helpfulness by ...


In [32]:
# Add your human evaluation labels
# Use only: "acceptable" or "not_acceptable"

human_labels = [
    "acceptable",       # Case 1: 2470126
    "acceptable",       # Case 2: 1705395
    "acceptable",       # Case 3: 2957011
    "not_acceptable"    # Case 4: 55058
]

# Add labels to the judge results
judge_results_df["human_label"] = human_labels

# Save the combined evaluation
judge_results_df.to_csv(
    "../results/reply_quality_human_eval.csv",
    index=False
)

print("Human evaluation saved successfully.")
print()

display(
    judge_results_df[
        [
            "tweet_id",
            "helpfulness",
            "groundedness",
            "unsupported_claims",
            "professional_tone",
            "overall_score",
            "human_label"
        ]
    ]
)

Human evaluation saved successfully.



,tweet_id,helpfulness,groundedness,unsupported_claims,professional_tone,overall_score,human_label
0,2470126,4,5,5,5,5,acceptable
1,1705395,5,5,5,5,5,acceptable
2,2957011,4,5,5,5,5,acceptable
3,55058,4,1,5,5,3,not_acceptable


In [33]:
from evaluation.human_agreement import calculate_human_agreement

# Extract human labels
human_labels = judge_results_df["human_label"].tolist()

# Extract LLM judge overall scores
judge_scores = judge_results_df["overall_score"].tolist()

# Calculate agreement
agreement_results = calculate_human_agreement(
    human_labels=human_labels,
    judge_scores=judge_scores,
    threshold=3
)

print("Human labels:")
print(human_labels)

print("\nLLM judge scores:")
print(judge_scores)

print("\nAgreement:", agreement_results["agreement"])

print("Cohen's Kappa:", agreement_results["cohen_kappa"])

Human labels:
['acceptable', 'acceptable', 'acceptable', 'not_acceptable']

LLM judge scores:
[5, 5, 5, 3]

Agreement: 0.75
Cohen's Kappa: 0.0


In [34]:
from evaluation.evaluate import evaluate_classifier
from evaluation.failure_analysis import get_top_failure_modes

# Evaluate the prototype classifier on the golden set
metrics, report, predictions = evaluate_classifier(
    classifier=prototype_classifier,
    golden_path="../data/golden/amazonhelp_golden.csv"
)

print("Classification metrics:")
print(metrics)

print("\nTop 5 failure modes:")
failure_modes = get_top_failure_modes(
    predictions,
    top_n=5,
    examples_per_mode=3
)

for i, mode in enumerate(failure_modes, start=1):

    print("\n" + "=" * 80)
    print(f"FAILURE MODE {i}")
    print("=" * 80)

    print(
        "True intent:",
        mode["true_intent"]
    )

    print(
        "Predicted intent:",
        mode["predicted_intent"]
    )

    print(
        "Error count:",
        mode["error_count"]
    )

    print("\nExamples:")

    for _, example in mode["examples"].iterrows():

        print(
            f'\nTweet ID: {example["tweet_id"]}'
        )

        print(
            f'Text: {example["text"]}'
        )

Classification metrics:
{'accuracy': 0.545, 'macro_f1': 0.4041748412801045, 'weighted_f1': 0.5284269149532309}

Top 5 failure modes:

FAILURE MODE 1
True intent: order_status
Predicted intent: other
Error count: 7

Examples:

Tweet ID: 493427
Text: @AmazonHelp Monday I think

Tweet ID: 1464284
Text: @AmazonHelp 8-9 nov

Tweet ID: 2954521
Text: @AmazonHelp 29/11/2017Votre colis est en cours de livraison

FAILURE MODE 2
True intent: delivery_issue
Predicted intent: other
Error count: 7

Examples:

Tweet ID: 930108
Text: @AmazonHelp Ich soll mich morgen nochmal melden, wenn die Sendung weiterhin nicht ausgeliefert werden sollte.

Tweet ID: 140066
Text: @AmazonHelp Voici : C20025897313 Parceque j'ai beau me plaindre à chaque fois (genre colis ouvert pour rentrer les articles 1 par 1 dans ma BaL), vous m'expediez de + en + par Amz Logistics et ça ne va pas mieux. Je vais bien finir par vous quitter hein…

Tweet ID: 133756
Text: @AmazonHelp Ja, die wissen auch nicht mehr weiter. Der Mitarbei

In [35]:
for i, mode in enumerate(failure_modes, start=1):

    print(f"\nFAILURE MODE {i}")
    print(f"True      : {mode['true_intent']}")
    print(f"Predicted : {mode['predicted_intent']}")
    print(f"Count     : {mode['error_count']}")

    for _, example in mode["examples"].iterrows():
        text = str(example["text"]).replace("\n", " ")
        print(f"- {example['tweet_id']}: {text}")


FAILURE MODE 1
True      : order_status
Predicted : other
Count     : 7
- 493427: @AmazonHelp Monday I think
- 1464284: @AmazonHelp 8-9 nov
- 2954521: @AmazonHelp 29/11/2017Votre colis est en cours de livraison

FAILURE MODE 2
True      : delivery_issue
Predicted : other
Count     : 7
- 930108: @AmazonHelp Ich soll mich morgen nochmal melden, wenn die Sendung weiterhin nicht ausgeliefert werden sollte.
- 140066: @AmazonHelp Voici : C20025897313 Parceque j'ai beau me plaindre à chaque fois (genre colis ouvert pour rentrer les articles 1 par 1 dans ma BaL), vous m'expediez de + en + par Amz Logistics et ça ne va pas mieux. Je vais bien finir par vous quitter hein…
- 133756: @AmazonHelp Ja, die wissen auch nicht mehr weiter. Der Mitarbeiter gerade hat gesagt, dass er nicht weiß, warum es nicht verschickt wird. 😂

FAILURE MODE 3
True      : complaint_or_escalation
Predicted : other
Count     : 6
- 1388465: @AmazonHelp Ça fait un moment que je remonte des dysfonctionnements... Pourtant auc

In [36]:
for i, mode in enumerate(failure_modes[3:5], start=4):

    print(f"\nFAILURE MODE {i}")
    print(f"True      : {mode['true_intent']}")
    print(f"Predicted : {mode['predicted_intent']}")
    print(f"Count     : {mode['error_count']}")

    for _, example in mode["examples"].iterrows():
        print(
            f"- {example['tweet_id']}: "
            f"{str(example['text']).replace(chr(10), ' ')}"
        )


FAILURE MODE 4
True      : other
Predicted : complaint_or_escalation
Count     : 5
- 117906: @AmazonHelp submitted my response.
- 1751594: @AmazonHelp 4. the seller by trying to locate the Contact the Seller button on my order, only to find there wasn't one. I therefore tried to leave a &gt;
- 38799: @AmazonHelp No reply to my last tweet. Thank you for replying now tho :) https://t.co/fYVeqY9Quq

FAILURE MODE 5
True      : order_status
Predicted : complaint_or_escalation
Count     : 4
- 182983: @AmazonHelp What is the status still no response
- 921197: @AmazonHelp i want to know when u r going to resolve it
- 1022898: @AmazonHelp I emailed amazon and spoke with someone named George and he said to reply to the email with my order number. I got an email back saying: https://t.co/L4QDMvt6Ub
